In [ ]:
%%writefile v2_nb4.py
# v2_nb4.py - shared helpers for Notebook 4 (results, figures, tables). READ-ONLY on NB3.
#  - freeze_check(): every NB4 cell first confirms that no frozen NB1-NB3 output has changed
#    since NB4 started (md5 manifest written once in NB4 cell 1)
#  - journal figure style (Okabe-Ito colour-blind-safe palette, 300 dpi PNG + vector PDF)
import os, glob, json, hashlib
import matplotlib
import matplotlib.pyplot as plt

P2, M2, FIG = "../data/processed_v2/", "../models_v2/", "../figures_v2/"
OUT4 = P2 + "nb4/"
ARMS = ["random_split", "temporal_split"]
ARM_LABEL = {"random_split": "Random-split test (n=556)",
             "temporal_split": "Temporal (2018-2022) test (n=1,404)"}
FROZEN = ["model_table_v2.csv", "splits_v2.csv", "cohort_final.csv",
          "cohort_delirium_timing.csv", "prep_*_t*.csv", "selected_features_*.csv",
          "train_oof_final_*.csv", "test_*.csv", "cv/*.csv", "pilot_labs/*.csv"]
FROZEN_M = ["final_*.joblib", "prep_*.joblib", "tuning_*.json", "locked_rules_*.json",
            "feature_pool_decision.json", "test_opened_log.json"]
MANIFEST = OUT4 + "frozen_manifest.json"

# Okabe-Ito palette (colour-blind safe)
C = {"model": "#0072B2", "stack": "#56B4E9", "lr": "#009E73", "sofa": "#E69F00",
     "apache": "#D55E00", "grey": "#999999", "black": "#000000", "pink": "#CC79A7",
     "yellow": "#F0E442"}


def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def _files():
    fs = [f for p in FROZEN for f in glob.glob(P2 + p)]
    fs += [f for p in FROZEN_M for f in glob.glob(M2 + p)]
    return sorted(set(fs))


def make_manifest():
    man = {os.path.relpath(f, ".."): _md5(f) for f in _files()}
    json.dump(man, open(MANIFEST, "w"), indent=1)
    return man


def freeze_check():
    """Stop if any frozen NB1-NB3 output changed after NB4 started."""
    man = json.load(open(MANIFEST))
    bad = [k for k, v in man.items()
           if not os.path.exists("../" + k) or _md5("../" + k) != v]
    assert not bad, f"FROZEN FILES CHANGED - stop and report: {bad[:5]}"
    return len(man)


def style():
    matplotlib.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 10,
        "axes.labelsize": 9, "legend.fontsize": 8, "xtick.labelsize": 8,
        "ytick.labelsize": 8, "axes.spines.top": False, "axes.spines.right": False,
        "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
        "pdf.fonttype": 42, "ps.fonttype": 42})


def save(fig, name):
    """Save a figure as 300-dpi PNG and editable vector PDF in figures_v2/."""
    os.makedirs(FIG, exist_ok=True)
    fig.savefig(FIG + name + ".png")
    fig.savefig(FIG + name + ".pdf")
    print(f"   saved figures_v2/{name}.png + .pdf")

In [ ]:
# ========= NB4 CELL 1b: setup, freeze the NB1-NB3 outputs, record software versions =========
# NB4 only READS the frozen results. The model, features and test results cannot change here.
import os, sys, json, platform, importlib
import numpy as np, pandas as pd
import v2_nb4
importlib.reload(v2_nb4)
from v2_nb4 import P2, M2, FIG, OUT4, ARMS, MANIFEST, make_manifest, freeze_check

os.makedirs(OUT4, exist_ok=True); os.makedirs(FIG, exist_ok=True)
ver = pd.read_csv(P2 + "nb3_verification.csv")
assert (ver.result == "FAIL").sum() == 0, "NB3 verification has FAILs - stop"
if not os.path.exists(MANIFEST):
    man = make_manifest()
    print(f"Frozen manifest created: {len(man)} NB1-NB3 output files fingerprinted (md5)")
print(f"Freeze check: {freeze_check()} frozen files unchanged | NB3 verification "
      f"{(ver.result == 'PASS').sum()} PASS, 0 FAIL")

# ---------------- software versions (TRIPOD+AI item: software) ----------------
mods = ["numpy", "pandas", "scipy", "sklearn", "xgboost", "catboost", "lightgbm", "optuna",
        "imblearn", "comorbidipy", "matplotlib", "shap", "statsmodels"]
vers = {"python": platform.python_version()}
for m in mods:
    try:
        vers[m] = importlib.import_module(m).__version__
    except Exception:
        vers[m] = "not installed"
json.dump(vers, open(OUT4 + "software_versions.json", "w"), indent=1)
print("\nSoftware versions (for the Methods section):")
print("   " + ", ".join(f"{k} {v}" for k, v in vers.items()))

# ---------------- frozen headline results (reference for every NB4 cell) ----------------
H = pd.read_csv(P2 + "test_headline_metrics.csv").set_index("arm")
log = json.load(open(M2 + "test_opened_log.json"))
print(f"\nFrozen test results (test opened once: {log['first_opened']}):")
for a in ARMS:
    h = H.loc[a]
    print(f"   {a:15s} AUC {h.auc:.3f} ({h.auc_lo:.3f}-{h.auc_hi:.3f}) | Brier {h.brier:.3f} | "
          f"slope {h.cal_slope:.2f} | E/O {h.expected_observed_ratio:.2f}")
need = ["shap", "statsmodels"]
miss = [m for m in need if vers[m] == "not installed"]
print("\nPackages still needed for later NB4 cells:",
      miss if miss else "none", "-> install with: %pip install " + " ".join(miss) if miss
      else "")
print("Folders ready: figures_v2/ and data/processed_v2/nb4/")

In [ ]:
# ========= NB4 CELL 2: TABLE 1 (baseline characteristics) + Table S1 (split balance) =========
# Raw, UN-imputed values (as measured). Continuous: median [IQR], Mann-Whitney U test;
# binary/categorical: n (%), chi-square test. Missing % shown for every measured variable.
# Table S1 compares train vs test in both arms with standardised mean differences (SMD;
# |SMD| < 0.1 = well balanced). Descriptive only - nothing here feeds the model.
import importlib, warnings
import numpy as np, pandas as pd
from scipy import stats
import v2_nb4
importlib.reload(v2_nb4)
from v2_nb4 import P2, OUT4, ARMS, freeze_check

warnings.filterwarnings("ignore")
freeze_check()
T = pd.read_csv(P2 + "model_table_v2.csv").set_index("stay_id")
CO = pd.read_csv(P2 + "cohort_final.csv", parse_dates=["intime", "outtime"]).set_index("stay_id")
DT = pd.read_csv(P2 + "cohort_delirium_timing.csv").set_index("stay_id")
CX = pd.read_csv(P2 + "clean_patient_context.csv").set_index("stay_id")
S = pd.read_csv(P2 + "splits_v2.csv").set_index("stay_id")
D = T.join(CO[["outtime", "intime"]]).join(DT, rsuffix="_dt").join(
    CX[[c for c in ["invasive_vent_24h", "niv_24h"] if c in CX]])
D["icu_los_days"] = (D.outtime - D.intime).dt.total_seconds() / 86400
D["age_lt60"] = (D.age_at_admission < 60).astype(int)
D["age_60_74"] = D.age_at_admission.between(60, 74.999).astype(int)
D["age_ge75"] = (D.age_at_admission >= 75).astype(int)
D["nee_users"] = D.nee_max.where(D.vaso_any == 1)       # dose among vasopressor users only
y = D.mortality_28d.astype(int)

# (label, column, kind): kind = cont | bin | cat
VARS = [("Demographics", None, "head"),
        ("Age, years", "age_at_admission", "cont"), ("Age <60", "age_lt60", "bin"),
        ("Age 60-74", "age_60_74", "bin"), ("Age >=75", "age_ge75", "bin"),
        ("Male sex", "sex_male", "bin"), ("Weight, kg", "weight_kg", "cont"),
        ("Cancer", None, "head"), ("Cancer type", "cancer_type", "cat"),
        ("Metastatic cancer", "elix_metacanc", "bin"),
        ("Solid tumour without metastasis", "elix_solidtum", "bin"),
        ("Lymphoma", "elix_lymph", "bin"),
        ("Comorbidities (Elixhauser)", None, "head"),
        ("van Walraven score (chronic)", "elix_vw_chronic", "cont"),
        ("Congestive heart failure", "elix_chf", "bin"),
        ("Chronic pulmonary disease", "elix_cpd", "bin"),
        ("Renal failure", "elix_rf", "bin"), ("Liver disease", "elix_ld", "bin"),
        ("Diabetes", "elix_diabetes", "bin"), ("Hypertension", "elix_hypertension", "bin"),
        ("Delirium", None, "head"), ("Delirium identified by", "delirium_route", "cat"),
        ("CAM-ICU positive by 24 h", "cam_positive_by_24h", "bin"),
        ("Hours to first CAM-ICU positive (if positive)", "cam_first_positive_h", "cont"),
        ("Severity, first 24 h", None, "head"),
        ("SOFA score", "sofa_total", "cont"), ("APACHE II score", "apache_ii_total", "cont"),
        ("Vasopressor use", "vaso_any", "bin"),
        ("Max norepinephrine-equivalent, mcg/kg/min (users)", "nee_users", "cont"),
        ("Invasive ventilation", "invasive_vent_24h", "bin"),
        ("Non-invasive ventilation", "niv_24h", "bin"),
        ("Physiology, first 24 h", None, "head"),
        ("Minimum GCS", "GCS_Total_min", "cont"), ("Minimum MAP, mmHg", "MAP_min", "cont"),
        ("Maximum heart rate, /min", "HR_max", "cont"),
        ("Maximum respiratory rate, /min", "RR_max", "cont"),
        ("Minimum SpO2, %", "SpO2_min", "cont"), ("Maximum temperature, C", "Temp_max", "cont"),
        ("Maximum lactate, mmol/L", "Lactate_max", "cont"),
        ("Maximum creatinine, mg/dL", "Creatinine_max", "cont"),
        ("Minimum platelets, x10^9/L", "Platelets_min", "cont"),
        ("Maximum bilirubin, mg/dL", "Bilirubin_max", "cont"),
        ("Maximum WBC, x10^9/L", "WBC_max", "cont"),
        ("Outcomes", None, "head"), ("ICU length of stay, days", "icu_los_days", "cont"),
        ("28-day mortality", "mortality_28d", "bin")]
absent = [c for _, c, k in VARS if k != "head" and c not in D.columns]
print("Variables not found (left out of the table):", absent if absent else "none")
VARS = [v for v in VARS if v[2] == "head" or v[1] in D.columns]


def fmt_cont(x):
    x = x.dropna()
    if not len(x):
        return "-"
    d = 2 if x.quantile(.75) < 1 else 1                  # small values (e.g. NEE): 2 decimals
    return f"{x.median():.{d}f} [{x.quantile(.25):.{d}f}-{x.quantile(.75):.{d}f}]"


def fmt_bin(x):
    x = x.dropna().astype(float)
    return f"{int(x.sum()):,} ({100 * x.mean():.1f})" if len(x) else "-"


def pval(col, kind):
    a, b = D.loc[y == 0, col], D.loc[y == 1, col]
    try:
        if kind == "cont":
            return stats.mannwhitneyu(a.dropna(), b.dropna()).pvalue
        tab = pd.crosstab(D[col], y)
        return stats.chi2_contingency(tab)[1] if tab.shape[0] > 1 else np.nan
    except ValueError:
        return np.nan


def pfmt(p):
    return "" if pd.isna(p) else ("<0.001" if p < 0.001 else f"{p:.3f}")


groups = {"Overall": np.ones(len(D), bool), "Survived": (y == 0).values,
          "Died within 28 days": (y == 1).values}
rows = [{"Characteristic": "Patients, n", **{g: f"{m.sum():,}" for g, m in groups.items()},
         "p": "", "Missing %": ""}]
for lab, col, kind in VARS:
    if kind == "head":
        rows.append({"Characteristic": lab.upper()}); continue
    base = D.loc[D.vaso_any == 1, col] if col == "nee_users" else D[col]
    miss = f"{100 * base.isna().mean():.1f}" if base.isna().any() else "0"
    if col == "mortality_28d":
        p = ""
    else:
        p = pfmt(pval(col, kind))
    if kind == "cat":
        rows.append({"Characteristic": lab + ", n (%)", "p": p, "Missing %": miss})
        for lev in D[col].dropna().value_counts().index:
            rows.append({"Characteristic": "   " + str(lev),
                         **{g: fmt_bin((D.loc[m, col] == lev).astype(float)
                                       .where(D.loc[m, col].notna()))
                            for g, m in groups.items()}})
    else:
        f = fmt_cont if kind == "cont" else fmt_bin
        unit = ", median [IQR]" if kind == "cont" else ", n (%)"
        rows.append({"Characteristic": lab + unit,
                     **{g: f(D.loc[m, col]) for g, m in groups.items()}, "p": p,
                     "Missing %": miss})
T1 = pd.DataFrame(rows).fillna("")
T1.to_csv(OUT4 + "table1_baseline.csv", index=False)


# ---------------- Table S1: train vs test balance in both arms (SMD) ----------------
def smd(a, b, kind):
    a, b = a.dropna().astype(float), b.dropna().astype(float)
    if kind == "bin":
        p1, p2 = a.mean(), b.mean()
        den = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)
        return (p1 - p2) / den if den > 0 else 0.0
    den = np.sqrt((a.var() + b.var()) / 2)
    return (a.mean() - b.mean()) / den if den > 0 else 0.0


srows = []
for lab, col, kind in VARS:
    if kind == "head" or col == "mortality_28d":
        continue
    levels = ([(f"{lab}: {l}", (D[col] == l).astype(float).where(D[col].notna()))
               for l in D[col].dropna().unique()] if kind == "cat"
              else [(lab, D[col].astype(float))])
    for name, x in levels:
        r = {"Characteristic": name}
        for arm in ARMS:
            tr = (S.loc[D.index, arm] == "train").values
            k = "bin" if kind in ("bin", "cat") else "cont"
            r[f"{arm}: train"] = fmt_cont(x[tr]) if k == "cont" else fmt_bin(x[tr])
            r[f"{arm}: test"] = fmt_cont(x[~tr]) if k == "cont" else fmt_bin(x[~tr])
            r[f"{arm}: SMD"] = round(smd(x[tr], x[~tr], k), 3)
        srows.append(r)
TS1 = pd.DataFrame(srows)
TS1.to_csv(OUT4 + "table_s1_split_balance.csv", index=False)

pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 48)
print("\nTABLE 1. Baseline characteristics (raw values, first 24 h)")
print(T1.to_string(index=False))
for arm in ARMS:
    big = TS1.loc[TS1[f"{arm}: SMD"].abs() >= 0.1, ["Characteristic", f"{arm}: SMD"]]
    print(f"\nTable S1 - {arm}: variables with |SMD| >= 0.1 (train vs test): "
          f"{len(big)} of {len(TS1)}")
    if len(big):
        print(big.to_string(index=False))
print("\nSaved: processed_v2/nb4/table1_baseline.csv, table_s1_split_balance.csv")

In [ ]:
# ========= NB4 CELL 3: main performance figures + calibration metrics (both test sets) =========
# Figure 2: ROC + precision-recall curves (final model vs Stacking, LR, SOFA, APACHE-II)
# Figure 3: calibration (LOESS-smoothed curve + decile points with 95% CI + risk histogram)
# Figure 4: decision curves (net benefit) for thresholds 5-80%
# Table : calibration metrics - slope, intercept, Brier, scaled Brier, ICI, E50, E90, Emax
#         (Austin & Steyerberg 2019; LOESS span 0.75). Uses ONLY the frozen test predictions.
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.proportion import proportion_confint
import v2_nb4, v2_final, v2_eval
for m in (v2_nb4, v2_final, v2_eval):
    importlib.reload(m)
from v2_nb4 import P2, OUT4, ARMS, ARM_LABEL, C, freeze_check, style, save
from v2_final import calibration_slope_intercept
from v2_eval import net_benefit, BRIER

warnings.filterwarnings("ignore")
freeze_check(); style()
PRED = {a: pd.read_csv(P2 + f"test_predictions_{a}.csv") for a in ARMS}
AUCT = {a: pd.read_csv(P2 + f"test_model_auc_{a}.csv").set_index("model") for a in ARMS}
SHOW = [("SoftVoting_calibrated", "Final model (Soft-Voting)", C["model"], "-", 2.0),
        ("Stacking", "Stacking", C["stack"], "--", 1.2),
        ("LR", "Logistic regression", C["lr"], "-.", 1.2),
        ("SOFA", "SOFA", C["sofa"], ":", 1.6),
        ("APACHE-II", "APACHE II", C["apache"], ":", 1.6)]


def lab(arm, key, name):
    r = AUCT[arm].loc[key]
    return f"{name}: {r.auc:.3f} ({r.lo:.3f}-{r.hi:.3f})"


# ------------------------------- Figure 2: ROC + PR -------------------------------
fig, ax = plt.subplots(2, 2, figsize=(9.2, 8.6))
for j, arm in enumerate(ARMS):
    d = PRED[arm]; y = d.y.values
    a = ax[0, j]
    for key, name, col, ls, lw in SHOW:
        fpr, tpr, _ = roc_curve(y, d["p_" + key])
        a.plot(fpr, tpr, color=col, ls=ls, lw=lw, label=lab(arm, key, name))
    a.plot([0, 1], [0, 1], color=C["grey"], lw=0.8)
    a.set(xlabel="1 - Specificity", ylabel="Sensitivity", xlim=(0, 1), ylim=(0, 1.01),
          title=f"{'AB'[j]}. ROC - {ARM_LABEL[arm]}")
    a.legend(loc="lower right", title="AUROC (95% CI)", title_fontsize=8, frameon=False)
    a = ax[1, j]
    for key, name, col, ls, lw in SHOW:
        pr, rc, _ = precision_recall_curve(y, d["p_" + key])
        a.plot(rc, pr, color=col, ls=ls, lw=lw,
               label=f"{name}: {average_precision_score(y, d['p_' + key]):.3f}")
    a.axhline(y.mean(), color=C["grey"], lw=0.8, label=f"Prevalence: {y.mean():.3f}")
    a.set(xlabel="Recall (sensitivity)", ylabel="Precision (PPV)", xlim=(0, 1), ylim=(0, 1.01),
          title=f"{'CD'[j]}. Precision-recall - {ARM_LABEL[arm]}")
    a.legend(loc="lower left", title="AUPRC", title_fontsize=8, frameon=False)
fig.tight_layout(); save(fig, "fig2_roc_pr"); plt.close(fig)


# -------------------------- calibration metrics (Austin & Steyerberg) --------------------------
def cal_metrics(y, p):
    sm = lowess(y, p, frac=0.75, it=0, return_sorted=False)
    err = np.abs(sm - p)
    slope, inter = calibration_slope_intercept(y, p)
    b = BRIER(y, p)
    return {"cal_slope": slope, "cal_intercept": inter, "brier": b,
            "scaled_brier": 1 - b / (y.mean() * (1 - y.mean())), "ICI": err.mean(),
            "E50": np.median(err), "E90": np.quantile(err, 0.9), "Emax": err.max(),
            "mean_predicted": p.mean(), "observed": y.mean()}


rows = []
for arm in ARMS:
    d = PRED[arm]
    for key, name, *_ in SHOW + [("va_p", "Final model (Venn-Abers)", 0, 0, 0)]:
        col = key if key == "va_p" else "p_" + key
        rows.append({"arm": arm, "model": name, **cal_metrics(d.y.values, d[col].values)})
CM = pd.DataFrame(rows)
CM.to_csv(OUT4 + "calibration_metrics.csv", index=False)

# --------------------------------- Figure 3: calibration ---------------------------------
fig, ax = plt.subplots(1, 2, figsize=(9.6, 4.9))
for j, arm in enumerate(ARMS):
    d = PRED[arm]; y, p = d.y.values, d.p_SoftVoting_calibrated.values
    a = ax[j]
    sm = lowess(y, p, frac=0.75, it=0)
    a.plot(sm[:, 0], sm[:, 1], color=C["model"], lw=2, label="LOESS calibration curve")
    q = pd.qcut(pd.Series(p).rank(method="first"), 10, labels=False)   # 10 equal-size groups
    g = pd.DataFrame({"y": y, "p": p, "q": q.values}).groupby("q")
    mp, ob, n = g.p.mean(), g.y.mean(), g.y.size()
    lo, hi = proportion_confint(g.y.sum(), n, method="wilson")         # stays inside 0-1
    a.errorbar(mp, ob, yerr=[np.clip(ob - lo, 0, None), np.clip(hi - ob, 0, None)], fmt="o", ms=4, color=C["black"], capsize=2,
               lw=0.8, label="Risk deciles (observed, 95% Wilson CI)")
    a.plot([0, 1], [0, 1], color=C["grey"], lw=0.8, ls="--", label="Perfect calibration")
    r = CM[(CM.arm == arm) & (CM.model == "Final model (Soft-Voting)")].iloc[0]
    a.text(0.03, 0.97, f"Slope {r.cal_slope:.2f}  Intercept {r.cal_intercept:.2f}\n"
           f"ICI {r.ICI:.3f}  E90 {r.E90:.3f}\nBrier {r.brier:.3f}  "
           f"Scaled Brier {r.scaled_brier:.3f}\nPredicted {r.mean_predicted:.3f} vs "
           f"observed {r.observed:.3f}", transform=a.transAxes, va="top", fontsize=7.5)
    h = a.inset_axes([0, 0, 1, 0.12], sharex=a)
    h.hist(p[y == 0], bins=40, range=(0, 1), color=C["grey"], alpha=0.7)
    h.hist(p[y == 1], bins=40, range=(0, 1), color=C["apache"], alpha=0.6)
    h.axis("off")
    a.set(xlim=(0, 1), ylim=(-0.15, 1.0), xlabel="Predicted 28-day mortality risk",
          ylabel="Observed 28-day mortality", title=f"{'AB'[j]}. {ARM_LABEL[arm]}")
    a.legend(loc="lower right", frameon=False, bbox_to_anchor=(1, 0.13))
fig.text(0.5, -0.02, "Bottom histograms: predicted risk of survivors (grey) and "
         "non-survivors (orange)", ha="center", fontsize=7.5)
fig.tight_layout(); save(fig, "fig3_calibration"); plt.close(fig)

# -------------------------------- Figure 4: decision curves --------------------------------
TH = np.round(np.arange(0.05, 0.81, 0.01), 2)
fig, ax = plt.subplots(1, 2, figsize=(9.6, 4.4))
nb_rows = []
for j, arm in enumerate(ARMS):
    d = PRED[arm]; y = d.y.values; a = ax[j]
    base = pd.DataFrame(net_benefit(y, d.p_SoftVoting_calibrated.values, TH))
    for key, name, col, ls, lw in [SHOW[0], SHOW[3], SHOW[4]]:
        nb = pd.DataFrame(net_benefit(y, d["p_" + key].values, TH))
        a.plot(TH, nb.model, color=col, ls=ls, lw=lw, label=name)
        nb_rows += [{"arm": arm, "model": name, "threshold": t, "net_benefit": v}
                    for t, v in zip(TH, nb.model)]
    a.plot(TH, base.treat_all, color=C["grey"], lw=1, label="Treat all")
    a.axhline(0, color=C["black"], lw=0.8, label="Treat none")
    a.set(xlim=(0.05, 0.8), ylim=(-0.05, max(0.42, y.mean() + 0.02)),
          xlabel="Threshold probability", ylabel="Net benefit",
          title=f"{'AB'[j]}. {ARM_LABEL[arm]}")
    a.legend(frameon=False)
fig.tight_layout(); save(fig, "fig4_decision_curve"); plt.close(fig)
NB = pd.DataFrame(nb_rows)
NB.to_csv(OUT4 + "decision_curve_all.csv", index=False)

# ------------------------------------------ print ------------------------------------------
pd.set_option("display.width", 200)
print("\nCALIBRATION METRICS (test sets; ICI/E50/E90 = absolute calibration error)")
print(CM.round(3).to_string(index=False))
print("\nNET BENEFIT at clinically relevant thresholds:")
pv = NB[NB.threshold.isin([0.2, 0.3, 0.4, 0.5])].pivot_table(
    index=["arm", "threshold"], columns="model", values="net_benefit")
for arm in ARMS:
    y = PRED[arm].y.values
    pv.loc[arm, "Treat all"] = [y.mean() - (1 - y.mean()) * t / (1 - t)
                                for t in pv.loc[arm].index]
print(pv.round(3).to_string())
for arm in ARMS:
    s = NB[(NB.arm == arm)].pivot(index="threshold", columns="model", values="net_benefit")
    y = PRED[arm].y.values
    ta = y.mean() - (1 - y.mean()) * s.index / (1 - s.index)
    best = s["Final model (Soft-Voting)"]
    use = (best > np.maximum(ta, 0)) & (best >= s["SOFA"]) & (best >= s["APACHE II"])
    print(f"{arm}: final model has the highest net benefit (vs treat-all/none, SOFA, APACHE II)"
          f" at {use.mean():.0%} of thresholds; range "
          f"{s.index[use].min():.2f}-{s.index[use].max():.2f}" if use.any() else "")
print("\nSaved: figures fig2_roc_pr, fig3_calibration, fig4_decision_curve (.png/.pdf);"
      " nb4/calibration_metrics.csv, nb4/decision_curve_all.csv")

In [ ]:
# ========= NB4 CELL 4: SHAP explanations + explanation stability (both test sets) =========
# SHAP of the FINAL Soft-Voting model, exact, in PROBABILITY space: interventional TreeSHAP for
# each base model (XGB, RF, CatBoost; background = 200 TRAINING patients of that arm), averaged
# with the soft-voting weights (1/3 each) -> contributions add up exactly to each patient's
# predicted risk (checked below). Isotonic calibration is a monotone step on top of this score,
# so it does not change which features drive the ranking.
# Stability (pre-specified): a predictor is STABLE if it was selected in >= 80% of the 25
# nested-CV folds AND its importance rank varies by <= 5 positions across 1,000 bootstrap
# resamples of the test patients (95% interval width of the rank).
# Second check: partial dependence (sklearn) for the top 6 features. Supplement: XGB SHAP
# interaction values (top pairs). Read-only on the frozen model and predictions.
import importlib, warnings
import numpy as np, pandas as pd, joblib, shap
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.inspection import partial_dependence
from statsmodels.nonparametric.smoothers_lowess import lowess
import v2_nb4, v2_select
for m in (v2_nb4, v2_select):
    importlib.reload(m)
from v2_nb4 import P2, M2, OUT4, ARMS, ARM_LABEL, C, freeze_check, style, save
from v2_select import feature_family, TPIFE_FAMILIES

warnings.filterwarnings("ignore")
freeze_check(); style()
N_BG, N_BOOT, TOP = 200, 1000, 15

NAMES = {"age_at_admission": "Age", "sex_male": "Male sex", "weight_kg": "Weight",
         "elix_metacanc": "Metastatic cancer", "elix_vw_chronic": "Elixhauser score (chronic)",
         "elix_solidtum": "Solid tumour", "elix_lymph": "Lymphoma",
         "sofa_total": "SOFA score", "apache_ii_total": "APACHE II score",
         "gcs_verbal_ETT_24h": "Intubated (GCS verbal not testable)",
         "resp_support_level": "Respiratory support level", "nee_max": "Max vasopressor dose",
         "vaso_hours": "Hours on vasopressor", "uo_ml_kg_h": "Urine output (mL/kg/h)",
         "csi_total": "Stability index (total)", "shock_index": "Shock index",
         "shock_index_max": "Shock index max", "apache_age_pts": "APACHE II age points",
         "tpife_mean_frac_abn": "Time abnormal, all vitals (TPIFE)"}
TOK = [("_late_minus_early_mean", " change 0-6h to 12-24h"), ("_early_mean", " mean 0-6h"),
       ("_mid_mean", " mean 6-12h"), ("_late_mean", " mean 12-24h"),
       ("_frac_abn", " time abnormal"), ("_frac_severe", " time severe"),
       ("_auc_dev", " abnormality burden"), ("_transitions_per24h", " state changes /24h"),
       ("_episodes_per24h", " abnormal episodes /24h"), ("_longest_episode_h", " longest episode"),
       ("_max_dev", " max deviation"), ("_ever_abn", " ever abnormal"),
       ("_ever_severe", " ever severe"), ("_last_state", " last state"), ("_slope", " slope"),
       ("_delta", " change"), ("_first", " first"), ("_last", " last"), ("_mean", " mean"),
       ("_min", " min"), ("_max", " max"), ("_std", " SD")]


def pretty(f):
    if f in NAMES:
        return NAMES[f]
    if f.startswith("osbi_"):
        return f[5:].replace("_", " ").capitalize() + " burden (OSBI)"
    if f.startswith("stab_"):
        v, k = f[5:].rsplit("_", 1)
        return f"{v} variability ({k.upper()})"
    if f.startswith("csi_"):
        return f"Stability index ({f[4:]})"
    if f.endswith("_first_episode_h"):
        return f[:-16].replace("_", " ") + " hours to first abnormal episode"
    for a, b in TOK:
        if f.endswith(a):
            return f[: -len(a)].replace("_", " ").replace("GCS Total", "GCS") + b
    return f.replace("_", " ")


def ens_shap(B, X, bg):
    """Exact probability-space SHAP of the soft-voting ensemble + per-model additivity check."""
    vote = B["final_uncalibrated"]
    w = 1 / len(vote.named_estimators_)
    phi, base, chk = 0.0, 0.0, {}
    for name, m in vote.named_estimators_.items():
        ex = shap.TreeExplainer(m, data=bg, feature_perturbation="interventional",
                                model_output="probability")
        v, ev = np.asarray(ex.shap_values(X, check_additivity=False)), np.ravel(ex.expected_value)
        if v.ndim == 3:
            v, ev = v[..., 1], ev[1]
        else:
            ev = ev[0]
        e = np.abs(v.sum(1) + ev - m.predict_proba(X)[:, 1])
        chk[name] = {"max_err": e.max(), "mean_err": e.mean(),
                     "n_err_gt_0.001": int((e > 1e-3).sum())}
        phi, base = phi + w * v, base + w * ev
    e = np.abs(phi.sum(1) + base - vote.predict_proba(X)[:, 1])
    chk["SoftVoting"] = {"max_err": e.max(), "mean_err": e.mean(),
                         "n_err_gt_0.001": int((e > 1e-3).sum())}
    return phi, base, pd.DataFrame(chk).T


# ---------------- nested-CV explanation stability (cell 5 files, training folds) ----------------
cvs = pd.read_csv(P2 + "cv/cv_shap.csv")
cvs["norm"] = cvs.mean_abs_shap / cvs.groupby(["repeat", "fold", "model"]).mean_abs_shap \
    .transform("sum")                                       # each model's SHAP share in the fold
fold_imp = cvs.groupby(["repeat", "fold", "feature"]).norm.mean().reset_index()
fold_imp["rank"] = fold_imp.groupby(["repeat", "fold"]).norm.rank(ascending=False)
n_folds = fold_imp[["repeat", "fold"]].drop_duplicates().shape[0]
cv_stab = pd.DataFrame({
    "cv_selected_pct": fold_imp.groupby("feature").size() / n_folds,
    "cv_top10_pct": fold_imp[fold_imp["rank"] <= 10].groupby("feature").size() / n_folds,
    "cv_median_rank": fold_imp.groupby("feature")["rank"].median()}).fillna(0)

RES, SH = {}, {}
rng = np.random.default_rng(42)
T = pd.read_csv(P2 + "model_table_v2.csv").set_index("stay_id")
for arm in ARMS:
    B = joblib.load(M2 + f"final_{arm}.joblib"); f = B["features"]
    tr = pd.read_csv(P2 + f"prep_{arm}_train.csv")
    te = pd.read_csv(P2 + f"prep_{arm}_test.csv")
    bg = tr[f].sample(N_BG, random_state=42)
    phi, base, chk = ens_shap(B, te[f], bg)
    print(f"{arm}: SHAP additivity check (|sum of SHAP + baseline - predicted risk|):")
    print(chk.to_string(float_format=lambda x: f"{x:.2e}"))
    err = chk.loc["SoftVoting", "max_err"]
    # pre-set tolerance: worst patient within 1 percentage point, average within 0.1 point
    assert err < 0.01 and chk.loc["SoftVoting", "mean_err"] < 0.001, "SHAP additivity failed"
    SH[arm] = (phi, te, f, B)
    pd.DataFrame(phi, columns=f).assign(stay_id=te.stay_id.values).to_csv(
        OUT4 + f"shap_test_{arm}.csv", index=False)
    imp = np.abs(phi).mean(0)
    # bootstrap the test patients: importance CI and rank stability
    bi = np.empty((N_BOOT, len(f)))
    for b in range(N_BOOT):
        bi[b] = np.abs(phi[rng.integers(0, len(phi), len(phi))]).mean(0)
    br = (-bi).argsort(1).argsort(1) + 1
    d = pd.DataFrame({"feature": f, "label": [pretty(x) for x in f],
                      "family": [feature_family(x) for x in f],
                      "mean_abs_shap": imp, "ci_lo": np.percentile(bi, 2.5, 0),
                      "ci_hi": np.percentile(bi, 97.5, 0),
                      "rank": (-imp).argsort().argsort() + 1,
                      "rank_lo": np.percentile(br, 2.5, 0), "rank_hi": np.percentile(br, 97.5, 0),
                      "direction_rho": [spearmanr(te[c], phi[:, i]).correlation
                                        for i, c in enumerate(f)]})
    d["tpife"] = d.family.isin(TPIFE_FAMILIES)
    if arm == "random_split":                     # nested CV was run on the random arm only
        d = d.merge(cv_stab, left_on="feature", right_index=True, how="left").fillna(
            {"cv_selected_pct": 0, "cv_top10_pct": 0})
        d["stable"] = (d.cv_selected_pct >= 0.8) & (d.rank_hi - d.rank_lo <= 5)
    d = d.sort_values("rank")
    d.to_csv(OUT4 + f"shap_importance_{arm}.csv", index=False)
    RES[arm] = (d, base, err)

# --------------------------- Figure 5: beeswarm + importance (random) ---------------------------
def beeswarm_bar(arm, name, letter):
    phi, te, f, _ = SH[arm]; d = RES[arm][0]
    top = d.feature.head(TOP).tolist()[::-1]
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 5.8), sharey=True,
                           gridspec_kw={"width_ratios": [1.35, 1], "wspace": 0.30})
    a = ax[0]
    for i, c in enumerate(top):
        v, x = phi[:, f.index(c)], te[c].values
        lo_, hi_ = np.nanpercentile(x, [5, 95])
        col = np.clip((x - lo_) / (hi_ - lo_ + 1e-12), 0, 1)
        jit = np.random.default_rng(i).normal(0, 0.09, len(v))
        sc = a.scatter(v, i + jit, c=col, cmap="coolwarm", s=5, alpha=0.75, lw=0)
    a.axvline(0, color=C["grey"], lw=0.8)
    a.set_yticks(range(len(top))); a.set_yticklabels([pretty(c) for c in top])
    a.set(xlabel="SHAP value (change in predicted 28-day mortality risk)",
          title=f"{letter[0]}. Effect of each predictor - {ARM_LABEL[arm]}")
    cax = a.inset_axes([1.01, 0.30, 0.018, 0.40])            # small bar beside panel A
    cb = fig.colorbar(sc, cax=cax, ticks=[0, 1])
    cb.ax.set_yticklabels(["low", "high"], fontsize=7)
    cb.ax.set_title("Feature\nvalue", fontsize=7)
    a.set_ylim(-0.6, len(top) - 0.4)
    a = ax[1]
    dd = d.head(TOP).iloc[::-1]
    cols = [C["pink"] if t else C["model"] for t in dd.tpife]
    a.barh(range(len(dd)), dd.mean_abs_shap, color=cols,
           xerr=[dd.mean_abs_shap - dd.ci_lo, dd.ci_hi - dd.mean_abs_shap],
           error_kw={"lw": 0.8, "capsize": 2})
    a.tick_params(axis="y", left=False)
    a.set(xlabel="Mean |SHAP| (95% bootstrap CI)",
          title=f"{letter[1]}. Global importance")
    a.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=C["pink"]),
                      plt.Rectangle((0, 0), 1, 1, color=C["model"])],
             labels=["TPIFE / stability / organ-burden", "Other features"],
             frameon=False, loc="upper right", bbox_to_anchor=(1.0, -0.09), ncol=2)
    fig.subplots_adjust(left=0.24, right=0.98, bottom=0.16, top=0.93)
    save(fig, name); plt.close(fig)


beeswarm_bar("random_split", "fig5_shap", "AB")
beeswarm_bar("temporal_split", "figS2_shap_temporal", "AB")

# ----------------- Figure 6: SHAP dependence + PDP check, top 6 features (random) -----------------
phi, te, f, B = SH["random_split"]; d = RES["random_split"][0]
top6 = d.feature.head(6).tolist()
raw = T.loc[te.stay_id.values]
fig, ax = plt.subplots(2, 3, figsize=(11, 6.9)); ax = ax.ravel()
pdp_rows = []
for k, c in enumerate(top6):
    a, x, v = ax[k], te[c].values, phi[:, f.index(c)]
    imput = raw[c].isna().values if c in raw.columns else np.zeros(len(x), bool)
    binary = len(np.unique(x)) <= 2
    xj = x + np.random.default_rng(k).normal(0, 0.03, len(x)) if binary else x
    a.scatter(xj[~imput], v[~imput], s=6, color=C["model"], alpha=0.5, lw=0)
    if imput.any():
        a.scatter(xj[imput], v[imput], s=8, facecolors="none", edgecolors=C["grey"], lw=0.6)
    if not binary:
        sm = lowess(v, x, frac=0.5, it=0)
        a.plot(sm[:, 0], sm[:, 1], color=C["black"], lw=1.5)
    pd_ = partial_dependence(B["final_uncalibrated"], te[f], [c], kind="average",
                             grid_resolution=25)
    gx, gy = pd_["grid_values"][0], pd_["average"][0]
    pdp_rows += [{"feature": c, "value": u, "pdp": w} for u, w in zip(gx, gy)]
    gy_c = gy - np.interp(x, gx, gy).mean()      # centred on the patients, like SHAP values
    if binary:
        a.plot(gx, gy_c, "D", color=C["apache"], ms=6)
        a.set_xticks([0, 1]); a.set_xticklabels(["No", "Yes"])
    else:
        a.plot(gx, gy_c, color=C["apache"], lw=1.3, ls="--")
    a.axhline(0, color=C["grey"], lw=0.6)
    a.set(xlabel=pretty(c), ylabel="Effect on 28-day risk" if k % 3 == 0 else "",
          title=f"{'ABCDEF'[k]}. {pretty(c)}")
H = [plt.Line2D([], [], marker="o", ls="", color=C["model"], ms=4, label="SHAP, measured"),
     plt.Line2D([], [], marker="o", ls="", mfc="none", mec=C["grey"], ms=4,
                label="SHAP, imputed value"),
     plt.Line2D([], [], color=C["black"], lw=1.5, label="SHAP trend (LOWESS)"),
     plt.Line2D([], [], color=C["apache"], lw=1.3, ls="--", label="Partial dependence")]
fig.legend(handles=H, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.04, 1, 1)); save(fig, "fig6_dependence_pdp"); plt.close(fig)
pd.DataFrame(pdp_rows).to_csv(OUT4 + "pdp_top6_random_split.csv", index=False)

# --------------- supplement: XGB interaction values (path-dependent, log-odds) ---------------
xgbm = B["final_uncalibrated"].named_estimators_["XGB"]
iv = np.abs(shap.TreeExplainer(xgbm).shap_interaction_values(te[f])).mean(0)
np.fill_diagonal(iv, 0)
iu = np.triu_indices(len(f), 1)
INT = pd.DataFrame({"feature_1": [pretty(f[i]) for i in iu[0]],
                    "feature_2": [pretty(f[j]) for j in iu[1]],
                    "mean_abs_interaction": 2 * iv[iu]}).sort_values(
    "mean_abs_interaction", ascending=False)
INT.head(20).to_csv(OUT4 + "shap_interactions_xgb_top20.csv", index=False)

# ------------------------------------------ print ------------------------------------------
pd.set_option("display.width", 230); pd.set_option("display.max_colwidth", 42)
for arm in ARMS:
    d, base, err = RES[arm]
    print("=" * 100, f"\n{arm.upper()}: SHAP of the final model on {len(SH[arm][0])} test patients "
          f"| baseline risk {base:.3f} | max additivity error {err:.1e}\n" + "=" * 100)
    cols = ["rank", "label", "family", "mean_abs_shap", "ci_lo", "ci_hi", "rank_lo", "rank_hi",
            "direction_rho"] + (["cv_selected_pct", "cv_top10_pct", "stable"]
                                if arm == "random_split" else [])
    print(d[cols].round(4).to_string(index=False))
    sh = d.groupby("family").mean_abs_shap.sum() / d.mean_abs_shap.sum()
    print("\nShare of total importance by feature family:")
    print((100 * sh.sort_values(ascending=False)).round(1).to_string())
    print(f"TPIFE + stability + organ-burden families together: "
          f"{100 * d.loc[d.tpife, 'mean_abs_shap'].sum() / d.mean_abs_shap.sum():.1f}% "
          f"of importance ({int(d.tpife.sum())} of {len(d)} features)")
    if arm == "random_split":
        print(f"Stable predictors (pre-specified rule): {int(d.stable.sum())} of {len(d)} -> "
              f"{d.loc[d.stable, 'label'].tolist()}")
print("\ndirection_rho > 0: higher value -> higher predicted risk; < 0: higher value -> lower")
print("\nTop 10 feature interactions (XGB component, supplement):")
print(INT.head(10).round(4).to_string(index=False))
print("\nSaved figures: fig5_shap, figS2_shap_temporal, fig6_dependence_pdp (.png/.pdf)")
print("Saved tables: nb4/shap_importance_<arm>.csv, shap_test_<arm>.csv, "
      "pdp_top6_random_split.csv, shap_interactions_xgb_top20.csv")

In [ ]:
# ========= NB4 CELL 5: risk tiers + uncertainty-aware tiers + clinical decision pathway =========
# Everything here uses rules LOCKED before the test set was opened (NB3 cell 7, Part A):
#   tiers = tertiles of TRAINING out-of-fold calibrated risk; "uncertain" = the patient's
#   Venn-Abers interval [p0, p1] crosses a tier cut-off. Nothing is re-chosen here.
# Outputs: Table 3 (tiers), Table 4 (uncertainty-aware tiers), Table 5 (decision pathway),
#          Figure 7 (tier calibration, interval width, confident vs uncertain).
# Trend test: logistic regression of death on tier (0/1/2), Wald p. Read-only.
import importlib, warnings, json
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import spearmanr, mannwhitneyu
from statsmodels.stats.proportion import proportion_confint
import v2_nb4
importlib.reload(v2_nb4)
from v2_nb4 import P2, M2, OUT4, ARMS, ARM_LABEL, C, freeze_check, style, save

warnings.filterwarnings("ignore")
freeze_check(); style()
TIERS = ["Low", "Intermediate", "High"]
TCOL = {"Low": C["lr"], "Intermediate": C["sofa"], "High": C["apache"]}
T = pd.read_csv(P2 + "model_table_v2.csv").set_index("stay_id")


def mort(y):
    y = np.asarray(y)
    if len(y) == 0:
        return np.nan, np.nan, np.nan
    lo, hi = proportion_confint(y.sum(), len(y), method="wilson")
    return y.mean(), lo, hi


def fmt(m, lo, hi):
    return "-" if np.isnan(m) else f"{100 * m:.1f} ({100 * lo:.1f}-{100 * hi:.1f})"


T3, T4, P = [], [], {}
for arm in ARMS:
    L = json.load(open(M2 + f"locked_rules_{arm}.json"))
    B = joblib.load(M2 + f"final_{arm}.joblib")
    d = pd.read_csv(P2 + f"test_predictions_{arm}.csv")
    d["tier"] = pd.Categorical(d.tier, TIERS, ordered=True)
    d["certain"] = ~d.uncertain_tier.astype(bool)
    raw = T.loc[d.stay_id, [c for c in B["features"] if c in T.columns]]
    d["n_imputed"] = raw.isna().sum(1).values             # model inputs that were imputed
    P[arm] = (d, L)
    # ---------------- Table 3: tiers ----------------
    for t in TIERS:
        g = d[d.tier == t]
        m, lo, hi = mort(g.y)
        T3.append({"arm": arm, "tier": t,
                   "risk_range": {"Low": f"< {L['tier_cut_low']:.1%}",
                                  "Intermediate": f"{L['tier_cut_low']:.1%} - "
                                                  f"{L['tier_cut_high']:.1%}",
                                  "High": f">= {L['tier_cut_high']:.1%}"}[t],
                   "n": len(g), "pct_patients": 100 * len(g) / len(d), "deaths": int(g.y.sum()),
                   "observed_mortality_pct (95% CI)": fmt(m, lo, hi), "obs": m, "lo": lo,
                   "hi": hi, "mean_predicted_pct": 100 * g.p_SoftVoting_calibrated.mean(),
                   "pct_uncertain": 100 * (~g.certain).mean(),
                   "median_VA_width": g.va_width.median()})
    # ---------------- Table 4: uncertainty-aware tiers ----------------
    for t in TIERS:
        for cert, lab in [(True, "confident"), (False, "uncertain")]:
            g = d[(d.tier == t) & (d.certain == cert)]
            m, lo, hi = mort(g.y)
            T4.append({"arm": arm, "category": f"{t} risk - {lab}", "tier": t, "certain": cert,
                       "n": len(g), "pct_patients": 100 * len(g) / len(d),
                       "deaths": int(g.y.sum()),
                       "observed_mortality_pct (95% CI)": fmt(m, lo, hi), "obs": m, "lo": lo,
                       "hi": hi, "mean_predicted_pct": 100 * g.p_SoftVoting_calibrated.mean()
                       if len(g) else np.nan,
                       "median_imputed_inputs": g.n_imputed.median() if len(g) else np.nan})
T3, T4 = pd.DataFrame(T3), pd.DataFrame(T4)
T3.drop(columns=["obs", "lo", "hi"]).to_csv(OUT4 + "table3_risk_tiers.csv", index=False)
T4.drop(columns=["obs", "lo", "hi"]).to_csv(OUT4 + "table4_uncertainty_tiers.csv",
                                            index=False)

# ---------------- trend test + what drives uncertainty ----------------
TR = []
for arm in ARMS:
    d, _ = P[arm]
    codes = d.tier.cat.codes.astype(float).values
    if np.unique(codes).size > 1:
        fit = sm.Logit(d.y.values, np.column_stack([np.ones(len(d)), codes])).fit(disp=0)
        or_step, p_tr = np.exp(fit.params[1]), fit.pvalues[1]
    else:
        or_step, p_tr = np.nan, np.nan
    hl = d[d.tier == "High"].y.mean() / d[d.tier == "Low"].y.mean()
    rho, prho = spearmanr(d.n_imputed, d.va_width)
    TR.append({"arm": arm, "OR_per_tier_step": or_step,
               "trend_p": p_tr, "mortality_ratio_high_vs_low": hl,
               "rho_width_vs_imputed_inputs": rho, "p_width_vs_imputed": prho,
               "median_width_died": d.va_width[d.y == 1].median(),
               "median_width_survived": d.va_width[d.y == 0].median(),
               "width_died_vs_survived_p": mannwhitneyu(d.va_width[d.y == 1],
                                                        d.va_width[d.y == 0]).pvalue,
               "median_width": d.va_width.median(),
               "width_IQR": f"{d.va_width.quantile(.25):.3f}-{d.va_width.quantile(.75):.3f}"})
TR = pd.DataFrame(TR)
TR.to_csv(OUT4 + "tier_trend_and_uncertainty.csv", index=False)

# ---------------- Table 5: clinical decision pathway (suggested, not validated) ----------------
ACT = {"Low": ("Routine ICU care; standard delirium bundle (ABCDEF); daily CAM-ICU; "
               "re-run the model if the clinical course changes"),
       "Intermediate": ("Closer neurological observation (CAM-ICU each shift); review "
                        "precipitating causes (sepsis, drugs, metabolic); early discussion "
                        "of goals of care with patient/family"),
       "High": ("Senior ICU and oncology review within 24 h; full delirium management; "
                "early goals-of-care and palliative-care involvement given cancer "
                "prognosis; intensify monitoring")}
UNC = ("Model estimate uncertain (interval crosses a tier boundary): rely on clinical "
       "judgement and re-assess at 48 h; do not use the tier alone")
rows = []
for t in TIERS:
    r = {"tier": t, "suggested_action": ACT[t]}
    for arm in ARMS:
        x = T3[(T3.arm == arm) & (T3.tier == t)].iloc[0]
        r[f"risk_range_{arm}"] = x.risk_range
        r[f"observed_mortality_{arm}"] = x["observed_mortality_pct (95% CI)"]
        r[f"pct_uncertain_{arm}"] = round(x.pct_uncertain, 1)
    rows.append(r)
rows.append({"tier": "Any tier, uncertain", "suggested_action": UNC})
T5 = pd.DataFrame(rows)
T5.to_csv(OUT4 + "table5_decision_pathway.csv", index=False)

# ---------------- Figure 7 ----------------
fig, ax = plt.subplots(1, 3, figsize=(12.5, 4.4))
a = ax[0]
for j, arm in enumerate(ARMS):
    s = T3[T3.arm == arm]
    xs = np.arange(3) + (j - 0.5) * 0.36
    a.bar(xs, s.obs.fillna(0) * 100, width=0.34, color=[TCOL[t] for t in TIERS],
          alpha=0.95 if j == 0 else 0.45, edgecolor="black", lw=0.5,
          hatch=None if j == 0 else "..",
          yerr=[(s.obs - s.lo).clip(lower=0).fillna(0) * 100,
                (s.hi - s.obs).clip(lower=0).fillna(0) * 100], capsize=3,
          error_kw={"lw": 0.8})
    a.scatter(xs, s.mean_predicted_pct, marker="D", color="black", s=18, zorder=3)
a.set_xticks(range(3)); a.set_xticklabels(TIERS)
a.set(ylabel="28-day mortality (%)", ylim=(0, 100),
      title="A. Observed mortality by risk tier")
a.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc="grey", ec="black"),
                  plt.Rectangle((0, 0), 1, 1, fc="grey", alpha=0.45, ec="black", hatch=".."),
                  plt.Line2D([], [], marker="D", ls="", color="black", ms=4)],
         labels=["Random-split test (observed, 95% CI)", "Temporal test (observed, 95% CI)",
                 "Mean predicted risk"], frameon=False, fontsize=7, loc="upper left")
a = ax[1]
d, L = P["random_split"]
data = [d.loc[d.tier == t, "va_width"].values for t in TIERS]
bp = a.boxplot(data, widths=0.5, patch_artist=True, showfliers=False,
               medianprops={"color": "black", "lw": 1.5})
for p_, t in zip(bp["boxes"], TIERS):
    p_.set_facecolor(TCOL[t]); p_.set_alpha(0.6)
for i, v in enumerate(data):
    a.scatter(np.random.default_rng(i).normal(i + 1, 0.06, len(v)), v, s=4, color="black",
              alpha=0.3, lw=0)
a.set_xticks([1, 2, 3]); a.set_xticklabels(TIERS)
a.set(ylabel="Venn-Abers interval width", title="B. Prediction uncertainty by tier (random)")
a = ax[2]
s = T4[T4.arm == "random_split"]
for k, cert in enumerate([True, False]):
    q = s[s.certain == cert]
    xs = np.arange(3) + (k - 0.5) * 0.36
    a.bar(xs, q.obs.fillna(0) * 100, width=0.34, color=[TCOL[t] for t in TIERS],
          alpha=0.95 if cert else 0.35, edgecolor="black", lw=0.5, hatch=None if cert else "//",
          yerr=[(q.obs - q.lo).clip(lower=0).fillna(0) * 100,
                (q.hi - q.obs).clip(lower=0).fillna(0) * 100], capsize=3,
          error_kw={"lw": 0.8})
    for x_, n_ in zip(xs, q.n):
        a.text(x_, 101, f"n={n_}", ha="center", fontsize=6.5, va="bottom")
a.set_xticks(range(3)); a.set_xticklabels(TIERS)
a.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc="grey", ec="black"),
                  plt.Rectangle((0, 0), 1, 1, fc="grey", alpha=0.35, ec="black", hatch="//")],
         labels=["Confident", "Uncertain (interval crosses a cut-off)"], frameon=False,
         fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=2)
a.set(ylabel="28-day mortality (%)", ylim=(0, 108))
a.set_title("C. Confident vs uncertain (random)", pad=14)
fig.tight_layout(); save(fig, "fig7_risk_tiers_uncertainty"); plt.close(fig)

# ------------------------------------------ print ------------------------------------------
pd.set_option("display.width", 230); pd.set_option("display.max_colwidth", 60)
print("TABLE 3. Risk tiers (cut-offs fixed on training data before the test was opened)")
print(T3.drop(columns=["obs", "lo", "hi"]).round(2).to_string(index=False))
print("\nTABLE 4. Uncertainty-aware risk tiers")
print(T4.drop(columns=["obs", "lo", "hi", "tier", "certain"]).round(2).to_string(index=False))
print("\nTrend across tiers and drivers of uncertainty:")
pf = lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
print(TR.assign(trend_p=TR.trend_p.map(pf), p_width_vs_imputed=TR.p_width_vs_imputed.map(pf),
                width_died_vs_survived_p=TR.width_died_vs_survived_p.map(pf))
      .round(4).to_string(index=False))
print("\nTABLE 5. Clinical decision pathway (suggested actions - require prospective evaluation)")
for r in T5.itertuples():
    print(f" - {r.tier}: {r.suggested_action}")
print("\nSaved: fig7_risk_tiers_uncertainty (.png/.pdf); nb4/table3_risk_tiers.csv, "
      "table4_uncertainty_tiers.csv, table5_decision_pathway.csv, tier_trend_and_uncertainty.csv")

In [ ]:
# ========= NB4 CELL 6: fairness / subgroup forest plot + CAM-ICU-by-24h deployment analysis ======
# Part A (pre-specified subgroups from NB3 cell 7: cancer type, sex, age group, delirium by 24h):
#   forest plot of test AUC (95% bootstrap CI) + calibration-in-the-large (E/O) per subgroup;
#   heterogeneity of AUC across levels: Cochran's Q (inverse-variance, SE from the bootstrap CI).
#   'reliable' = >= 30 deaths and >= 30 survivors (fixed before the test was opened).
# Part B (pre-specified sensitivity analysis): patients CAM-ICU positive by 24 h = the population
#   in whom the model can actually be used at the 24-h prediction time. Full metrics + DeLong
#   vs SOFA / APACHE II inside this group, and the same for patients not positive by 24 h.
# Read-only on frozen predictions.
import importlib, warnings, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import v2_nb4, v2_eval, v2_final
for m in (v2_nb4, v2_eval, v2_final):
    importlib.reload(m)
from v2_nb4 import P2, M2, OUT4, ARMS, ARM_LABEL, C, freeze_check, style, save
from v2_eval import delong_test, delong_ci, boot_ci, threshold_metrics, net_benefit, AUC, BRIER
from v2_final import calibration_slope_intercept

warnings.filterwarnings("ignore")
freeze_check(); style()
SG = pd.read_csv(P2 + "test_subgroups.csv")
H = pd.read_csv(P2 + "test_headline_metrics.csv").set_index("arm")
VAR_LABEL = {"cancer_type": "Cancer type", "sex": "Sex", "age_group": "Age group",
             "delirium_by_24h": "Delirium timing"}
ORDER = {"age_group": ["<60", "60-74", ">=75"], "sex": ["female", "male"]}
pf = lambda p: "" if pd.isna(p) else ("<0.001" if p < 0.001 else f"{p:.3f}")

# ------------------------- Part A: heterogeneity (Cochran's Q) -------------------------
SG["se"] = (SG.auc_hi - SG.auc_lo) / (2 * 1.96)
SG["E_O"] = SG.mean_predicted / SG.obs_mortality
het = []
for (arm, var), g in SG.dropna(subset=["auc", "se"]).groupby(["arm", "subgroup"]):
    g = g[g.se > 0]
    if len(g) < 2:
        continue
    w = 1 / g.se ** 2
    pooled = (w * g.auc).sum() / w.sum()
    Q = (w * (g.auc - pooled) ** 2).sum()
    het.append({"arm": arm, "subgroup": var, "levels": len(g), "Q": Q, "df": len(g) - 1,
                "p_heterogeneity": stats.chi2.sf(Q, len(g) - 1),
                "auc_range": f"{g.auc.min():.3f}-{g.auc.max():.3f}"})
HET = pd.DataFrame(het)
HET.to_csv(OUT4 + "subgroup_heterogeneity.csv", index=False)

# ------------------------------- Figure 8: forest plot -------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 7.2), sharey=True)
rows, labels = [], []
for var in VAR_LABEL:
    levels = ORDER.get(var, sorted(SG[SG.subgroup == var].level.unique()))
    rows.append((var, None)); labels.append(VAR_LABEL[var])
    for lev in levels:
        rows.append((var, lev)); labels.append("   " + str(lev))
ypos = np.arange(len(rows))[::-1]
for j, arm in enumerate(ARMS):
    a = ax[j]
    s = SG[SG.arm == arm].set_index(["subgroup", "level"])
    for y_, (var, lev) in zip(ypos, rows):
        if lev is None or (var, lev) not in s.index:
            continue
        r = s.loc[(var, lev)]
        if pd.isna(r.auc):
            a.text(0.505, y_, "not estimable", va="center", fontsize=7, color=C["grey"])
            continue
        col = C["model"] if r.reliable else C["grey"]
        a.errorbar(r.auc, y_, xerr=[[r.auc - r.auc_lo], [r.auc_hi - r.auc]], fmt="s",
                   color=col, ms=3 + 5 * np.sqrt(r.n / s.n.max()), capsize=2, lw=1)
        a.text(1.03, y_, f"{int(r.n):>5} / {int(r.deaths):<4} {r.auc:.2f}   {r.E_O:.2f}",
               va="center", fontsize=7, family="monospace",
               transform=a.get_yaxis_transform())
    a.axvline(H.loc[arm, "auc"], color=C["apache"], ls="--", lw=1,
              label=f"Overall AUROC {H.loc[arm, 'auc']:.3f}")
    a.axvline(0.5, color=C["grey"], lw=0.6)
    a.set(xlim=(0.45, 1.0), xlabel="AUROC (95% bootstrap CI)",
          title=f"{'AB'[j]}. {ARM_LABEL[arm]}")
    a.text(1.03, 1.0, f"{'n':>5} / {'died':<4} {'AUC':<4}   E/O", fontsize=7,
           family="monospace", va="bottom", transform=a.transAxes)
    a.legend(loc="lower left", frameon=False, fontsize=7)
ax[0].set_yticks(ypos); ax[0].set_yticklabels(labels)
for t in ax[0].get_yticklabels():
    if not t.get_text().startswith("   "):
        t.set_fontweight("bold")
fig.text(0.01, 0.005, "Grey = fewer than 30 deaths or 30 survivors (estimate not reliable). "
         "E/O = mean predicted / observed mortality.", fontsize=7)
fig.subplots_adjust(left=0.17, right=0.84, wspace=0.62, top=0.94, bottom=0.08)
save(fig, "fig8_subgroups_forest"); plt.close(fig)

# --------------- Part B: CAM-ICU positive by 24 h (deployable population) ---------------
DT = pd.read_csv(P2 + "cohort_delirium_timing.csv").set_index("stay_id")
dep = []
for arm in ARMS:
    thr = json.load(open(M2 + f"locked_rules_{arm}.json"))["threshold"]
    d = pd.read_csv(P2 + f"test_predictions_{arm}.csv")
    d["cam24"] = DT.loc[d.stay_id, "cam_positive_by_24h"].values == 1
    for grp, m in [("CAM-ICU positive by 24 h", d.cam24), ("Not positive by 24 h", ~d.cam24),
                   ("All test patients", np.ones(len(d), bool))]:
        g = d[m.values if hasattr(m, "values") else m]
        y, pc = g.y.values, g.p_SoftVoting_calibrated.values
        a_, lo, hi = boot_ci(y, pc, AUC)
        slope, inter = calibration_slope_intercept(y, pc)
        tm = threshold_metrics(y, pc, thr)
        r = {"arm": arm, "group": grp, "n": len(g), "deaths": int(y.sum()),
             "mortality": y.mean(), "auc_final": a_, "auc_lo": lo, "auc_hi": hi,
             "brier": BRIER(y, pc), "cal_slope": slope, "cal_intercept": inter,
             "E_O": pc.mean() / y.mean(), "sensitivity": tm["sensitivity"],
             "specificity": tm["specificity"], "ppv": tm["ppv"], "npv": tm["npv"]}
        for other in ["SOFA", "APACHE-II", "Stacking", "LR"]:
            a1, a2, p = delong_test(y, g.p_SoftVoting.values, g["p_" + other].values)
            r[f"auc_{other}"], r[f"p_vs_{other}"] = a2, p
        nb = pd.DataFrame(net_benefit(y, pc, [0.3]))
        r["net_benefit_at_0.30"] = nb.model.iloc[0]
        r["net_benefit_treat_all_0.30"] = nb.treat_all.iloc[0]
        dep.append(r)
DEP = pd.DataFrame(dep)
DEP.to_csv(OUT4 + "table6_delirium24h_deployment.csv", index=False)
# difference in AUC between the two independent groups (z-test, bootstrap SEs)
dz = []
for arm in ARMS:
    a_ = DEP[(DEP.arm == arm) & (DEP.group == "CAM-ICU positive by 24 h")].iloc[0]
    b_ = DEP[(DEP.arm == arm) & (DEP.group == "Not positive by 24 h")].iloc[0]
    se = np.sqrt(((a_.auc_hi - a_.auc_lo) / 3.92) ** 2 + ((b_.auc_hi - b_.auc_lo) / 3.92) ** 2)
    z = (a_.auc_final - b_.auc_final) / se
    dz.append({"arm": arm, "auc_diff": a_.auc_final - b_.auc_final,
               "p_difference": 2 * stats.norm.sf(abs(z))})
DZ = pd.DataFrame(dz)

# ------------------------------------------ print ------------------------------------------
pd.set_option("display.width", 240); pd.set_option("display.max_colwidth", 30)
print("SUBGROUPS (test sets): AUROC, calibration-in-the-large (E/O), sensitivity/specificity")
cols = ["arm", "subgroup", "level", "n", "deaths", "auc", "auc_lo", "auc_hi", "E_O",
        "sensitivity", "specificity", "reliable"]
print(SG[cols].round(3).to_string(index=False))
print("\nHeterogeneity of AUROC across levels (Cochran's Q):")
print(HET.assign(p_heterogeneity=HET.p_heterogeneity.map(pf)).round(2).to_string(index=False))
print("\nTABLE 6. CAM-ICU positive by 24 h (deployable population) vs not")
show = ["arm", "group", "n", "deaths", "mortality", "auc_final", "auc_lo", "auc_hi",
        "auc_SOFA", "p_vs_SOFA", "auc_APACHE-II", "p_vs_APACHE-II", "auc_Stacking",
        "p_vs_Stacking", "brier", "cal_slope", "E_O", "sensitivity", "specificity",
        "net_benefit_at_0.30", "net_benefit_treat_all_0.30"]
out = DEP[show].copy()
for c in [c for c in show if c.startswith("p_vs")]:
    out[c] = out[c].map(pf)
print(out.round(3).T.to_string(header=False))
print("\nAUROC difference, CAM-ICU+ by 24 h minus not (independent groups, z-test):")
print(DZ.assign(p_difference=DZ.p_difference.map(pf)).round(3).to_string(index=False))
print("\nSaved: fig8_subgroups_forest (.png/.pdf); nb4/subgroup_heterogeneity.csv, "
      "table6_delirium24h_deployment.csv")

In [ ]:
# ========= NB4 CELL 7: paper tables - Table 2 (all models) + supplementary tables S2-S9 =========
# Collects every frozen NB3/NB4 result into publication tables (CSV + one Excel workbook).
# Nothing is re-estimated except AUPRC/Brier, which are re-computed from the frozen test
# predictions. Feature dictionary is regenerated with the authoritative feature_family()
# (the NB2 Step-10 dictionary mislabelled early/mid/late segment means as 'basic temporal').
import os, json, importlib, warnings
import numpy as np, pandas as pd, joblib
from sklearn.metrics import average_precision_score, brier_score_loss
import v2_nb4, v2_select
for m in (v2_nb4, v2_select):
    importlib.reload(m)
from v2_nb4 import P2, M2, OUT4, ARMS, freeze_check
from v2_select import feature_family, is_measurement, TPIFE_FAMILIES

warnings.filterwarnings("ignore")
freeze_check()
TAB = OUT4 + "tables/"
os.makedirs(TAB, exist_ok=True)
pf = lambda p: "" if pd.isna(p) else ("<0.001" if p < 0.001 else f"{p:.3f}")
ci = lambda a, lo, hi: f"{a:.3f} ({lo:.3f}-{hi:.3f})"
OUT = {}

# ------------------------------ Table 2: every model ------------------------------
MODELS = [("SoftVoting_calibrated", "Soft-Voting ensemble, isotonic (FINAL)", "SoftVoting"),
          ("SoftVoting_VennAbers", "Soft-Voting ensemble, Venn-Abers", None),
          ("Stacking", "Stacking ensemble", "Stacking"), ("XGB", "XGBoost", "XGB"),
          ("LightGBM", "LightGBM", "LightGBM"), ("CatBoost", "CatBoost", "CatBoost"),
          ("RF", "Random forest", "RF"), ("LR", "Logistic regression", "LR"),
          ("DT", "Decision tree", "DT"), ("MultiView_meta", "Multi-View (diagnostic)", None),
          ("SOFA", "SOFA score", "SOFA"), ("APACHE-II", "APACHE II score", "APACHE-II")]
cv = pd.read_csv(P2 + "cv/cv_summary.csv").query("pool == 'B_primary'").set_index("model")
rows = []
for key, name, cvkey in MODELS:
    r = {"Model": name}
    if cvkey in cv.index:
        r["Nested CV AUROC, mean (SD)"] = f"{cv.loc[cvkey, 'auc_mean']:.3f} " \
                                          f"({cv.loc[cvkey, 'auc_sd']:.3f})"
    for arm in ARMS:
        A = pd.read_csv(P2 + f"test_model_auc_{arm}.csv").set_index("model")
        Pd = pd.read_csv(P2 + f"test_predictions_{arm}.csv")
        col = "va_p" if key == "SoftVoting_VennAbers" else "p_" + key
        tag = "random" if arm == "random_split" else "temporal"
        if key in A.index:
            r[f"AUROC {tag} (95% CI)"] = ci(A.loc[key, "auc"], A.loc[key, "lo"], A.loc[key, "hi"])
        r[f"AUPRC {tag}"] = round(average_precision_score(Pd.y, Pd[col]), 3)
        calibrated = key in ("SoftVoting_calibrated", "SoftVoting_VennAbers", "SOFA",
                             "APACHE-II", "Stacking", "MultiView_meta")
        r[f"Brier {tag}"] = round(brier_score_loss(Pd.y, Pd[col]), 3) if calibrated else "n/a*"
    rows.append(r)
T2 = pd.DataFrame(rows).fillna("-")
OUT["Table2_models"] = T2

# ------------------ S2: pre-specified comparisons (CV t-tests + test DeLong) ------------------
tt = pd.read_csv(P2 + "cv/cv_corrected_ttests.csv").assign(
    setting="Nested CV (corrected resampled t-test)", p=lambda d: d.p_value.map(pf))
dl = pd.concat([pd.read_csv(P2 + f"test_delong_{a}.csv") for a in ARMS])
dl = dl.assign(setting=dl.arm.map({"random_split": "Random-split test (DeLong)",
                                   "temporal_split": "Temporal test (DeLong)"}),
               mean_auc_diff=dl["diff"].round(4), p=dl.p_delong.map(pf))
tt["scores"] = "out-of-fold AUROC per outer fold (25 folds)"
dl["scores"] = "uncalibrated Soft-Voting scores vs comparator"
S2 = pd.concat([tt[["setting", "comparison", "mean_auc_diff", "p", "scores"]],
                dl[["setting", "comparison", "mean_auc_diff", "p", "scores"]]])
OUT["S2_comparisons"] = S2

# ------------------------- S3: ablations (pools) CV + test -------------------------
cva = pd.read_csv(P2 + "cv/cv_summary.csv")
S3 = cva[cva.model.isin(["SoftVoting", "XGB", "LR", "Stacking"])].pivot(
    index="model", columns="pool", values="auc_mean").round(4)
S3.columns = [f"CV AUROC: {c}" for c in S3.columns]
for arm in ARMS:
    A = pd.read_csv(P2 + f"test_model_auc_{arm}.csv").set_index("model")
    tag = "random" if arm == "random_split" else "temporal"
    S3.loc["SoftVoting", f"Test {tag}: primary (uncalibrated)"] = round(A.loc["SoftVoting", "auc"], 4)
    S3.loc["SoftVoting", f"Test {tag}: no TPIFE (uncalibrated)"] = round(A.loc["SV_noTPIFE", "auc"], 4)
    S3.loc["SoftVoting", f"Test {tag}: with measurement (uncalibrated)"] = \
        round(A.loc["SV_with_measurement", "auc"], 4)
OUT["S3_ablations"] = S3.reset_index()

# -------------------------- S4: extra-labs pilot (training only) --------------------------
pil = pd.read_csv(P2 + "pilot_labs/pilot_cv_auc.csv")
S4 = pd.DataFrame({"Feature set": pil.columns, "CV AUROC mean": pil.mean().round(4).values,
                   "SD": pil.std().round(4).values,
                   "Gain vs current": (pil.mean() - pil.iloc[:, 0].mean()).round(4).values,
                   "Folds better": [(pil[c] > pil.iloc[:, 0]).mean() for c in pil.columns]})
S4["Decision (pre-set rule: add only if gain >= 0.005)"] = \
    np.where(S4["Gain vs current"].iloc[1] >= 0.005, "add labs", "keep current model")
OUT["S4_lab_pilot"] = S4

# ------------------------------ S5: complete-case sensitivity ------------------------------
S5 = pd.concat([pd.read_csv(P2 + f"test_complete_case_{a}.csv") for a in ARMS]).round(3)
S5 = S5[["arm", "n_complete_case", "auc_complete_case_model", "auc_main_model_same_patients"]]
OUT["S5_complete_case"] = S5

# ------------------------------ S6: tuned hyperparameters ------------------------------
hp = []
for arm in ARMS:
    tj = json.load(open(M2 + f"tuning_{arm}.json"))
    for mname, prm in tj["best_params"].items():
        hp.append({"arm": arm, "model": mname,
                   "hyperparameters": "; ".join(f"{k}={round(v, 4) if isinstance(v, float) else v}"
                                                for k, v in prm.items()),
                   "imbalance_handling": tj["imbalance_method"]})
OUT["S6_hyperparameters"] = pd.DataFrame(hp)

# --------------- S7: final features (family, CV selection %, SHAP rank) ---------------
freq = pd.read_csv(P2 + "cv/cv_feature_selection_frequency.csv").set_index("feature")
fs = []
for arm in ARMS:
    B = joblib.load(M2 + f"final_{arm}.joblib")
    sh = pd.read_csv(OUT4 + f"shap_importance_{arm}.csv").set_index("feature")
    for f in B["features"]:
        fs.append({"arm": arm, "feature": f, "label": sh.loc[f, "label"],
                   "family": feature_family(f), "TPIFE family": feature_family(f) in TPIFE_FAMILIES,
                   "SHAP rank (test)": int(sh.loc[f, "rank"]),
                   "mean |SHAP|": round(sh.loc[f, "mean_abs_shap"], 4),
                   "nested-CV selection %": round(100 * freq.selection_frequency.get(f, 0), 0)
                   if arm == "random_split" else np.nan})
S7 = pd.DataFrame(fs).sort_values(["arm", "SHAP rank (test)"])
OUT["S7_final_features"] = S7

# ------------------- S8: full feature dictionary with corrected families -------------------
FD = pd.read_csv(P2 + "feature_dictionary_v2.csv")
FD["family"] = FD.feature.map(lambda f: "demographics/cancer" if f == "cancer_type"
                              else feature_family(f))
FD["family_changed_from_step10"] = FD.family != FD.group
FD["measurement_process_feature"] = FD.feature.map(is_measurement)   # excluded from pool B
OUT["S8_feature_dictionary"] = FD.drop(columns="group")

# ------------------------------ S9: Multi-View diagnostic ------------------------------
mv = []
for arm in ARMS:
    A = pd.read_csv(P2 + f"test_model_auc_{arm}.csv").set_index("model")
    for k in [i for i in A.index if i.startswith("View_")] + ["MultiView_meta", "SoftVoting"]:
        name = {"MultiView_meta": "Multi-View meta-learner",
                "SoftVoting": "Final Soft-Voting, uncalibrated scores (reference)"}.get(k, k.replace("View_", "") +
                                                                  " view only")
        mv.append({"arm": arm, "view": name,
                   "AUROC (95% CI)": ci(A.loc[k, "auc"], A.loc[k, "lo"], A.loc[k, "hi"])})
OUT["S9_multiview"] = pd.DataFrame(mv)

# ------------------------------------------ save ------------------------------------------
for k, v in OUT.items():
    v.to_csv(TAB + f"{k}.csv", index=False)
try:
    with pd.ExcelWriter(TAB + "all_tables.xlsx") as xw:
        for k, v in OUT.items():
            v.to_excel(xw, sheet_name=k[:31], index=False)
    xl = "and all_tables.xlsx (one sheet per table)"
except Exception as e:
    xl = f"(Excel not written: {type(e).__name__})"

pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
print("TABLE 2. Discrimination of all models (test sets; nested CV on random-split training)")
print(T2.to_string(index=False))
print("* Brier not shown for class-weighted uncalibrated models (probabilities not calibrated "
      "by design; compared on discrimination only).")
print("  Isotonic calibration maps risks onto a step function; the tied values lower AUPRC slightly "
      "and change AUROC by at most about 0.005 versus the uncalibrated scores.")
print("  Pre-specified DeLong tests (S2), ablations (S3) and the Multi-View comparison (S9) use the "
      "uncalibrated Soft-Voting scores (fixed for the DeLong tests before the test set was opened).")
print("\nS2. Pre-specified comparisons:"); print(S2.to_string(index=False))
print("\nS3. Ablations:"); print(S3.reset_index().to_string(index=False))
print("\nS4. Extra-labs pilot:"); print(S4.to_string(index=False))
print("\nS5. Complete-case sensitivity:"); print(S5.to_string(index=False))
print("\nS7. Final features (random arm):")
print(S7[S7.arm == "random_split"].drop(columns="arm").to_string(index=False))
print(f"\nS8. Feature dictionary: {len(FD)} features; family corrected for "
      f"{int(FD.family_changed_from_step10.sum())} features vs the Step-10 file; "
      f"{int(FD.measurement_process_feature.sum())} measurement-process features flagged")
print(FD.family.value_counts().to_string())
print("\nS9. Multi-View:"); print(OUT["S9_multiview"].to_string(index=False))
print(f"\nSaved: data/processed_v2/nb4/tables/<table>.csv {xl}")

In [ ]:
# ========= NB4 CELL 8: sample-size adequacy (Riley criteria) + TRIPOD+AI checklist map =========
# Part A - TRIPOD+AI item 10: is the development sample large enough?
#   Riley et al. (BMJ 2020) minimum-sample-size criteria for a binary outcome:
#   (i) expected shrinkage >= 0.9, (ii) optimism in Nagelkerke R2 <= 0.05,
#   (iii) overall risk estimated within +/-0.05. Cox-Snell R2 is derived from the nested-CV
#   C-statistic by simulation (Riley et al., Stat Med 2021). P = number of model parameters
#   (final selected predictors). Also reports events per predictor (EPP).
# Part B - TRIPOD+AI (Collins et al., BMJ 2024;385:e078378): every item mapped to where our
#   pipeline provides the evidence, and what is still needed for the manuscript.
import importlib, warnings, json
import numpy as np, pandas as pd, joblib
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression
import v2_nb4
importlib.reload(v2_nb4)
from v2_nb4 import P2, M2, OUT4, ARMS, freeze_check

warnings.filterwarnings("ignore")
freeze_check()


def r2cs_from_c(c, prev, n=1_000_000, seed=1):
    """Cox-Snell R2 implied by a C-statistic and prevalence (binormal simulation)."""
    rng = np.random.default_rng(seed)
    y = rng.random(n) < prev
    x = rng.normal(np.where(y, np.sqrt(2) * norm.ppf(c), 0.0), 1.0)
    m = LogisticRegression(C=1e6, max_iter=1000).fit(x.reshape(-1, 1), y)
    p = np.clip(m.predict_proba(x.reshape(-1, 1))[:, 1], 1e-12, 1 - 1e-12)
    ll1 = np.sum(y * np.log(p) + (~y) * np.log(1 - p))
    ll0 = n * (prev * np.log(prev) + (1 - prev) * np.log(1 - prev))
    return 1 - np.exp(-2 * (ll1 - ll0) / n)


S = pd.read_csv(P2 + "splits_v2.csv").set_index("stay_id")
T = pd.read_csv(P2 + "model_table_v2.csv", usecols=["stay_id", "mortality_28d"]) \
    .set_index("stay_id")
cv = pd.read_csv(P2 + "cv/cv_summary.csv").query("pool == 'B_primary'").set_index("model")
c_stat = cv.loc["SoftVoting", "auc_mean"]
rows = []
for arm in ARMS:
    y = T.loc[S.index[S[arm] == "train"], "mortality_28d"]
    n, ev, phi = len(y), int(y.sum()), y.mean()
    P = len(joblib.load(M2 + f"final_{arm}.joblib")["features"])
    r2 = r2cs_from_c(c_stat, phi)
    r2max = 1 - (phi ** phi * (1 - phi) ** (1 - phi)) ** 2
    n1 = P / ((0.9 - 1) * np.log(1 - r2 / 0.9))
    s2 = r2 / (r2 + 0.05 * r2max)
    n2 = P / ((s2 - 1) * np.log(1 - r2 / s2))
    n3 = (1.96 / 0.05) ** 2 * phi * (1 - phi)
    need = int(np.ceil(max(n1, n2, n3)))
    rows.append({"arm": arm, "n_train": n, "events": ev, "prevalence": round(phi, 3),
                 "parameters_P": P, "assumed_C": round(c_stat, 3), "R2_CoxSnell": round(r2, 3),
                 "n_crit1_shrinkage": int(np.ceil(n1)), "n_crit2_optimism": int(np.ceil(n2)),
                 "n_crit3_intercept": int(np.ceil(n3)), "n_required": need,
                 "adequate": n >= need, "EPP": round(ev / P, 1)})
SS = pd.DataFrame(rows)
SS.to_csv(OUT4 + "sample_size_riley.csv", index=False)

# --------------------------------- Part B: TRIPOD+AI map ---------------------------------
R, W, A, NA = "Ready - evidence in pipeline", "Write in manuscript", "ACTION NEEDED", \
    "Not applicable - state why"
CHK = [
 ("1", "Title", "Development + validation of ML model; cancer ICU patients with delirium; "
  "28-day mortality", W, "State 'development and temporal validation'"),
 ("2", "Abstract", "Follow TRIPOD+AI for Abstracts", W,
  "Include intended-use result (CAM-ICU+ by 24 h AUROC 0.83/0.82)"),
 ("3a", "Background", "Context, rationale, existing models (SOFA/APACHE II)", W, ""),
 ("3b", "Background", "Target population, intended purpose and users", W,
  "ICU clinicians; adults with cancer and delirium detected by 24 h; prediction at 24 h"),
 ("3c", "Background", "Known health inequalities", W,
  "Discuss sex/age; race/ethnicity not analysed -> limitation"),
 ("4", "Objectives", "Development and validation objectives", W, ""),
 ("5a", "Data sources", "MIMIC-IV v3.1, single centre (BIDMC)", R, "NB1"),
 ("5b", "Data dates", "Admissions 2008-2022 (anchor years), era split <=2017 / >2017", R,
  "NB1 calendar era"),
 ("6a", "Setting", "ICUs of one tertiary academic centre, Boston, USA", W, ""),
 ("6b", "Eligibility", "Cancer ICD (2,052 codes) + delirium (ICD or CAM-ICU), first ICU "
  "stay, age >=18, ICU stay >=24 h", R, "NB1 attrition + icd_code_appendix"),
 ("6c", "Treatments", "Vasopressors/ventilation in first 24 h used as predictors; later "
  "treatment not modelled", W, "Discuss treatment-limitation decisions in cancer"),
 ("7", "Data preparation", "Plausibility ranges, unit conversion, artefact rules, windows", R,
  "NB2 cleaning_log.csv"),
 ("8a", "Outcome", "Death within 28 days of ICU admission (hospital + patients.dod)", R, ""),
 ("8b", "Outcome assessors", "Objective outcome (death)", NA, "No subjective assessment"),
 ("8c", "Outcome blinding", "Death recorded independently of predictors", NA, ""),
 ("9a", "Predictor choice", "391 candidate features; ELIX acute categories excluded "
  "(no POA flag)", R, "NB2 Step 10; S8 dictionary"),
 ("9b", "Predictor definitions", "All windows/thresholds defined", R,
  "S8 dictionary + tpife_thresholds.csv"),
 ("9c", "Predictor assessors", "CAM-ICU nursing assessment (inclusion only)", W,
  "CAM-ICU not a predictor; describe routine nursing assessment"),
 ("10", "Sample size", "All eligible patients; Riley criteria + EPP", R,
  "Part A of this cell"),
 ("11", "Missing data", "20% rule (lactate 30%), MICE fitted on training only, "
  "complete-case sensitivity", R, "NB3 cell 2; S5"),
 ("12a", "Data use/partition", "Random 80/20 + temporal split; nested 5x5 CV", R, "NB3"),
 ("12b", "Predictor handling", "Clipping 0.5-99.5%, scaling, CSI, one-hot", R, "v2_prep.py"),
 ("12c", "Model type/building", "6 learners + ensembles; stability SHAP-RFE; Optuna; "
  "pre-written selection rule", R, "NB3 cells 3-6; S6"),
 ("12d", "Heterogeneity across clusters", "Single centre: temporal era as cluster", W,
  "Report random vs temporal performance"),
 ("12e", "Performance measures", "AUROC, AUPRC, Brier, calibration slope/ICI, DCA, DeLong",
  R, "NB3 cell 7; NB4 cell 3"),
 ("12f", "Model updating", "No updating after test", NA, "Test opened once"),
 ("12g", "Prediction calculation", "Isotonic-calibrated Soft-Voting probability", R, ""),
 ("13", "Class imbalance", "Class weights (SMOTEENN tested, rejected); isotonic "
  "recalibration", R, "NB3 cell 4"),
 ("14", "Fairness", "Pre-specified subgroups: cancer type, sex, age, delirium timing; "
  "Cochran's Q", R, "NB4 cell 6, Fig 8"),
 ("15", "Model output", "Calibrated risk + Venn-Abers interval; Youden threshold from "
  "training OOF; locked tertile tiers", R, "NB3 cell 6-7; NB4 cell 5"),
 ("16", "Development vs evaluation", "Table S1 SMD train vs test (both splits)", R,
  "NB4 cell 2"),
 ("17", "Ethics", "MIMIC-IV: de-identified, IRB-approved (BIDMC/MIT), consent waived; "
  "PhysioNet credentialed access + DUA", W, "Cite PhysioNet MIMIC-IV approval"),
 ("18a", "Funding", "Funding source / none", A, "Add statement"),
 ("18b", "Conflicts of interest", "Declarations", A, "Add statement"),
 ("18c", "Protocol", "analysis_plan_v2.json + pre-specified rules; no formal protocol", W,
  "Provide analysis plan as supplement"),
 ("18d", "Registration", "Not registered", W, "State 'not registered'"),
 ("18e", "Data sharing", "MIMIC-IV available via PhysioNet to credentialed users", W, ""),
 ("18f", "Code sharing", "Notebooks + v2_*.py modules", A,
  "Create public GitHub repository (no patient data) and cite it"),
 ("19", "Patient/public involvement", "None", W, "State 'no PPI'"),
 ("20a", "Participant flow", "Attrition 74,503 -> 2,777; 1,060 deaths", R,
  "NB1 attrition_counts_final.csv -> flow diagram (Figure 1)"),
 ("20b", "Participant characteristics", "Table 1", R, "NB4 cell 2"),
 ("20c", "Development vs evaluation comparison", "Table S1", R, "NB4 cell 2"),
 ("21", "Participants/events per analysis", "Train/test n and deaths both splits", R,
  "Tables 1, 2, S1"),
 ("22", "Full model specification", "Saved models + hyperparameters + feature list", A,
  "Share final_<arm>.joblib or code to rebuild; S6, S7"),
 ("23a", "Performance with CIs incl. subgroups", "Table 2, Figs 2-4, 8, Table 6", R,
  "NB3 cell 7; NB4 cells 3, 6"),
 ("23b", "Heterogeneity across clusters", "Random vs temporal test", W, ""),
 ("24", "Model updating results", "None", NA, ""),
 ("25", "Interpretation", "Incl. fairness (hematologic lower AUROC)", W, ""),
 ("26", "Limitations", "Single centre; delirium any time in stay; no external validation; "
  "variability features may reflect monitoring intensity; race not analysed", W,
  "Use methods-log disclosure list"),
 ("27a", "Poor-quality/missing inputs", "Training-fitted imputer + clipping ship with model; "
  "uncertainty flag", W, ""),
 ("27b", "User requirements", "24 h of routine EHR data; CAM-ICU documented", W, ""),
 ("27c", "Next steps", "External validation (e.g. eICU-CRD), prospective evaluation of "
  "decision pathway", W, ""),
]
TR = pd.DataFrame(CHK, columns=["item", "topic", "what_we_report", "status", "where_or_action"])
TR.to_csv(OUT4 + "tripod_ai_checklist.csv", index=False)

pd.set_option("display.width", 230); pd.set_option("display.max_colwidth", 70)
print("PART A. Sample size (Riley et al. criteria, development data)")
print(SS.T.to_string(header=False))
print("Note: criteria use the final number of predictors; selection from many candidates is "
      "accounted for by redoing selection inside every nested-CV fold and by the untouched "
      "test sets.")
print("\nPART B. TRIPOD+AI checklist status:")
print(TR.status.value_counts().to_string())
print("\nItems needing ACTION before submission:")
print(TR[TR.status == A][["item", "topic", "where_or_action"]].to_string(index=False))
print("\nSaved: nb4/tripod_ai_checklist.csv, nb4/sample_size_riley.csv")

In [ ]:
# ========= NB4 CELL 9: POST-HOC comparison of the final model with logistic regression ==========
# and the Multi-View diagnostic model (NOT pre-specified -> reported as post-hoc in the paper).
# Why: on the temporal test set the simple logistic regression (LR) had a higher AUROC (0.775) than
#      the final Soft-Voting model (0.759); reviewers will ask whether this difference is real.
# What:  DeLong test + paired stratified bootstrap (2,000) CI of the AUROC difference, on the SAME
#        frozen test predictions (nothing is refitted). Done for:
#          - all test patients, and
#          - the intended-use group (CAM-ICU positive by 24 h)
#        Final model compared both as uncalibrated scores (as in the pre-specified DeLong tests)
#        and as the calibrated output (what a clinician would see).
# Holm correction across the post-hoc tests of each arm is also reported.
# Read-only. Output: data/processed_v2/nb4/posthoc_model_comparisons.csv
import importlib, warnings, json
import numpy as np, pandas as pd
from scipy.stats import rankdata
import v2_nb4, v2_eval
for m in (v2_nb4, v2_eval):
    importlib.reload(m)
from v2_nb4 import P2, OUT4, ARMS, freeze_check
from v2_eval import delong_test, AUC

warnings.filterwarnings("ignore")
n_frozen = freeze_check()
DT = pd.read_csv(P2 + "cohort_delirium_timing.csv").set_index("stay_id")
LR_PARAMS = {arm: json.load(open(v2_nb4.M2 + f"tuning_{arm}.json"))["best_params"]["LR"] for arm in ARMS}


def fast_auc(y, p):
    """Mann-Whitney AUROC with mid-ranks for ties (identical to sklearn roc_auc_score)."""
    r = rankdata(p); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

COMPARE = [  # (final-model column, comparator column, label)
    ("p_SoftVoting", "p_LR", "Soft-Voting (uncalibrated) vs logistic regression"),
    ("p_SoftVoting_calibrated", "p_LR", "Soft-Voting (calibrated, final) vs logistic regression"),
    ("p_SoftVoting", "p_MultiView_meta", "Soft-Voting (uncalibrated) vs Multi-View"),
]


def paired_boot_diff(y, p1, p2, n_boot=2000, seed=42):
    """Stratified paired bootstrap CI of AUC(p1) - AUC(p2) (same resampled patients)."""
    y, p1, p2 = np.asarray(y), np.asarray(p1), np.asarray(p2)
    rng = np.random.default_rng(seed)
    i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]
    d = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.r_[rng.choice(i1, len(i1)), rng.choice(i0, len(i0))]
        d[b] = fast_auc(y[idx], p1[idx]) - fast_auc(y[idx], p2[idx])
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float((d > 0).mean())


def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for k, i in enumerate(order):
        run = max(run, min(1.0, (m - k) * p[i])); adj[i] = run
    return adj


rows = []
for arm in ARMS:
    T = pd.read_csv(P2 + f"test_predictions_{arm}.csv")
    assert T.stay_id.is_unique and set(T.y.unique()) <= {0, 1}
    for c in ["p_SoftVoting", "p_LR"]:   # fast AUC must equal the sklearn AUC used everywhere else
        assert abs(fast_auc(T.y.values, T[c].values) - AUC(T.y.values, T[c].values)) < 1e-12
    for c1, c2, _ in COMPARE:
        assert c1 in T and c2 in T, f"missing column {c1} or {c2} in test_predictions_{arm}.csv"
        assert T[[c1, c2]].notna().all().all(), f"missing predictions in {arm}"
    T["cam24"] = T.stay_id.map(DT.cam_positive_by_24h).astype(float)
    assert T.cam24.notna().all(), "delirium-timing missing for some test patients"
    groups = {"All test patients": T, "CAM-ICU positive by 24 h": T[T.cam24 == 1]}
    for gname, G in groups.items():
        y = G.y.values
        for c1, c2, label in COMPARE:
            a1, a2, p = delong_test(y, G[c1].values, G[c2].values)
            lo, hi, share = paired_boot_diff(y, G[c1].values, G[c2].values)
            rows.append({"arm": arm, "population": gname, "n": len(G), "deaths": int(y.sum()),
                         "comparison": label, "auc_final": round(a1, 4), "auc_comparator": round(a2, 4),
                         "diff": round(a1 - a2, 4), "diff_lo": round(lo, 4), "diff_hi": round(hi, 4),
                         "p_delong": p, "boot_share_final_better": round(share, 3)})
R = pd.DataFrame(rows)
R["p_holm_within_arm"] = np.nan
for arm in ARMS:
    m = R.arm == arm
    R.loc[m, "p_holm_within_arm"] = holm(R.loc[m, "p_delong"].values)
R["analysis"] = "post-hoc (not pre-specified)"
R.to_csv(OUT4 + "posthoc_model_comparisons.csv", index=False)

# ------------------------------------ print ------------------------------------
pf = lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
print(f"Frozen files unchanged: {n_frozen}\n")
print("POST-HOC AUROC comparisons (DeLong; 95% CI of difference from 2,000 paired bootstraps)")
for arm in ARMS:
    print(f"\n=== {arm} ===")
    for _, r in R[R.arm == arm].iterrows():
        print(f"  [{r.population}, n={r.n}, deaths={r.deaths}] {r.comparison}:\n"
              f"      {r.auc_final:.3f} vs {r.auc_comparator:.3f}  diff {r['diff']:+.3f} "
              f"({r.diff_lo:+.3f} to {r.diff_hi:+.3f})  p={pf(r.p_delong)}  Holm p={pf(r.p_holm_within_arm)}")
    print(f"  LR tuned hyperparameters ({arm}): {LR_PARAMS[arm]}")

print("\nINTERPRETATION (automatic, check by eye):")
for arm in ARMS:
    r = R[(R.arm == arm) & (R.population == "All test patients") &
          (R.comparison.str.startswith("Soft-Voting (uncalibrated) vs logistic"))].iloc[0]
    sig = r.p_holm_within_arm < 0.05
    better = "final model" if r["diff"] > 0 else "logistic regression"
    print(f"  {arm}: {better} higher by {abs(r['diff']):.3f}; "
          + ("statistically significant after Holm correction." if sig
             else "NOT statistically significant after Holm correction (models perform similarly)."))
print("\nSaved: data/processed_v2/nb4/posthoc_model_comparisons.csv")

In [ ]:
# ========= NB4 CELL 10: the "GCS paradox" of the comparator study (He et al., 2025) ==============
# He Y, et al. Asia-Pac J Oncol Nurs 2025;12:100760 (MIMIC-IV, cancer + delirium, 28-day mortality)
# reported that a HIGH Glasgow Coma Scale (GCS) score was the strongest predictor of HIGHER mortality
# ("contradictory pattern"), and hypothesised that patients with a high GCS at ICU admission later
# deteriorate more during delirium. They did not test this.
# We test three explanations with the raw (non-imputed) first-24-h values of the WHOLE cohort.
# This is an explanatory association analysis, not part of model building or testing
# (no model is refitted, no prediction changes) -> reported as post-hoc.
#   H0  Replication: is a higher first GCS linked to higher mortality in our data? (unadjusted)
#   H1  Intubation / sedation artefact: intubated patients get a verbal score of 1 ("No
#       Response-ETT") and a low total GCS; many are post-operative and survive. Does the GCS-death
#       association change when intubation in the first 24 h is taken into account?
#   H2  Trajectory (He et al.'s hypothesis + roadmap): does the GCS trajectory in the first 24 h
#       (TPIFE / temporal features: slope, late-minus-early mean) explain the association?
#       Ceiling effect warning: a GCS of 15 can only stay or fall, so "high baseline -> more fall"
#       is partly arithmetic; we therefore also look at the trajectory WITHIN baseline groups.
#   Adjusted model adds age, metastatic cancer and SOFA without its CNS part (the CNS part IS GCS).
# Outputs: nb4/gcs_paradox_models.csv, nb4/gcs_paradox_groups.csv, figures_v2/figS3_gcs_paradox
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
import v2_nb4, v2_eval
for m in (v2_nb4, v2_eval):
    importlib.reload(m)
from v2_nb4 import P2, OUT4, C, freeze_check, style, save
from v2_eval import AUC

warnings.filterwarnings("ignore")
n_frozen = freeze_check(); style()

# ------------------------------------------- data -------------------------------------------
MT = pd.read_csv(P2 + "model_table_v2.csv")
CO = pd.read_csv(P2 + "cohort_final.csv")[["stay_id", "mortality_28d"]]
SP = pd.read_csv(P2 + "splits_v2.csv")[["stay_id", "temporal_split", "est_calendar_year"]]
need = ["stay_id", "mortality_28d", "GCS_Total_first", "GCS_Total_last", "GCS_Total_slope",
        "GCS_Total_late_minus_early_mean", "GCS_Total_min", "resp_support_level", "gcs_verbal_ETT_24h",
        "age_at_admission", "elix_metacanc", "sofa_total", "sofa_cns"]
miss_cols = [c for c in need if c not in MT]
assert not miss_cols, f"columns missing in model_table_v2.csv: {miss_cols}"
D = MT[need].merge(CO, on="stay_id", suffixes=("", "_cohort")).merge(SP, on="stay_id")
assert len(D) == len(MT) == len(CO), "row counts differ between model table / cohort / splits"
assert (D.mortality_28d == D.mortality_28d_cohort).all(), "outcome differs between files"

D["y"] = D.mortality_28d.astype(int)
D["gcs_first"] = D.GCS_Total_first
D["gcs_change"] = D.GCS_Total_last - D.GCS_Total_first            # last minus first, 0-24 h
D["intubated"] = ((D.resp_support_level >= 3) |
                  (D.gcs_verbal_ETT_24h.astype(str).str.lower().isin(["1", "1.0", "true"]))).astype(int)
D["sofa_noncns"] = D.sofa_total - D.sofa_cns
D["age10"] = D.age_at_admission / 10
D["metastatic"] = D.elix_metacanc.astype(int)
D["gcs_band"] = pd.cut(D.gcs_first, [2.5, 8.5, 12.5, 15.5], labels=["3-8", "9-12", "13-15"])
D["trajectory"] = np.select([D.gcs_change >= 2, D.gcs_change <= -2], ["Improving (>=+2)", "Worsening (<=-2)"],
                            "Stable (-1 to +1)")
D.loc[D.gcs_change.isna(), "trajectory"] = np.nan
assert D.gcs_first.dropna().between(3, 15).all(), "first GCS outside 3-15"

# --- the comparator study's Table 1 reports GCS mean 13.07 (SD 8.92): impossible on a 3-15 scale
#     (Bhatia-Davis inequality: SD <= sqrt((max - mean) * (mean - min)))
he_mean, he_sd = 13.07, 8.92
sd_max = np.sqrt((15 - he_mean) * (he_mean - 3))
ours = D.gcs_first.describe()

# ----------------------------------- regression models -----------------------------------
TRAJ = ["GCS_Total_slope", "GCS_Total_late_minus_early_mean"]
ADJ = ["age10", "metastatic", "sofa_noncns"]
CC = D.dropna(subset=["gcs_first", "intubated"] + TRAJ + ADJ).copy()   # one complete-case set for all
for c in TRAJ:                       # per 1 SD so the ORs are comparable
    CC[c + "_z"] = (CC[c] - CC[c].mean()) / CC[c].std()
TRAJ_Z = [c + "_z" for c in TRAJ]
MODELS = [
    ("M1 first GCS only (unadjusted)", ["gcs_first"]),
    ("M2 + intubation in first 24 h", ["gcs_first", "intubated"]),
    ("M3 + GCS trajectory (slope, late-minus-early)", ["gcs_first"] + TRAJ_Z),
    ("M4 + intubation + trajectory", ["gcs_first", "intubated"] + TRAJ_Z),
    ("M5 M4 + age, metastatic cancer, SOFA without CNS", ["gcs_first", "intubated"] + TRAJ_Z + ADJ),
]
rows, fits = [], {}
for name, X in MODELS:
    f = sm.Logit(CC.y, sm.add_constant(CC[X].astype(float))).fit(disp=0)
    fits[name] = f
    ci = f.conf_int()
    for term in X:
        rows.append({"model": name, "term": term, "OR": np.exp(f.params[term]),
                     "OR_lo": np.exp(ci.loc[term, 0]), "OR_hi": np.exp(ci.loc[term, 1]),
                     "p": f.pvalues[term], "n": int(f.nobs), "deaths": int(CC.y.sum()),
                     "auc": AUC(CC.y, f.predict()), "aic": f.aic})
MOD = pd.DataFrame(rows)


def lr_test(small, big):
    s, b = fits[small], fits[big]
    stat = 2 * (b.llf - s.llf); df = b.df_model - s.df_model
    return stat, df, stats.chi2.sf(stat, df)


LRT = {"intubation added to M1": lr_test(MODELS[0][0], MODELS[1][0]),
       "trajectory added to M1": lr_test(MODELS[0][0], MODELS[2][0]),
       "trajectory added to M2": lr_test(MODELS[1][0], MODELS[3][0])}

# interaction: does the GCS-death slope differ between intubated and not intubated?
CC["gcs_x_intub"] = CC.gcs_first * CC.intubated
fi = sm.Logit(CC.y, sm.add_constant(CC[["gcs_first", "intubated", "gcs_x_intub"]].astype(float))).fit(disp=0)
p_inter = fi.pvalues["gcs_x_intub"]
strat = []
for k, lab in [(0, "Not intubated"), (1, "Intubated")]:
    S = CC[CC.intubated == k]
    f = sm.Logit(S.y, sm.add_constant(S[["gcs_first"]].astype(float))).fit(disp=0)
    lo, hi = np.exp(f.conf_int().loc["gcs_first"])
    strat.append({"model": f"Stratum: {lab} (first GCS only)", "term": "gcs_first",
                  "OR": np.exp(f.params["gcs_first"]), "OR_lo": lo, "OR_hi": hi,
                  "p": f.pvalues["gcs_first"], "n": len(S), "deaths": int(S.y.sum()),
                  "auc": AUC(S.y, f.predict()) if 0 < S.y.sum() < len(S) else np.nan, "aic": f.aic})
# same in each era (robustness)
for era in ["train", "test"]:
    S = CC[CC.temporal_split == era]
    f = sm.Logit(S.y, sm.add_constant(S[["gcs_first"]].astype(float))).fit(disp=0)
    lo, hi = np.exp(f.conf_int().loc["gcs_first"])
    strat.append({"model": f"Era {'<=2017' if era == 'train' else '>2017'} (first GCS only)",
                  "term": "gcs_first", "OR": np.exp(f.params["gcs_first"]), "OR_lo": lo, "OR_hi": hi,
                  "p": f.pvalues["gcs_first"], "n": len(S), "deaths": int(S.y.sum()),
                  "auc": np.nan, "aic": f.aic})
MOD = pd.concat([MOD, pd.DataFrame(strat)], ignore_index=True)
MOD["interaction_p_gcs_x_intubation"] = p_inter
MOD.to_csv(OUT4 + "gcs_paradox_models.csv", index=False)


# ---------------------------------- descriptive groups ----------------------------------
def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


grp = []
for by in [["gcs_band"], ["intubated", "gcs_band"], ["gcs_band", "trajectory"]]:
    for key, g in D.dropna(subset=by).groupby(by, observed=True):
        key = key if isinstance(key, tuple) else (key,)
        lo, hi = wilson(g.y.sum(), len(g))
        r = {"grouping": " x ".join(by), "n": len(g), "deaths": int(g.y.sum()),
             "mortality": g.y.mean(), "lo": lo, "hi": hi,
             "median_gcs_change": g.gcs_change.median(),
             "pct_intubated": g.intubated.mean()}
        for k_, v_ in zip(by, key):
            r[k_] = v_
        grp.append(r)
GRP = pd.DataFrame(grp)
GRP.to_csv(OUT4 + "gcs_paradox_groups.csv", index=False)
# He et al.'s hypothesis: GCS change by baseline band (with ceiling-effect caveat)
kw = stats.kruskal(*[g.gcs_change.dropna() for _, g in D.groupby("gcs_band", observed=True)])

# ------------------------------------------ figure ------------------------------------------
BANDS = ["3-8", "9-12", "13-15"]
fig, ax = plt.subplots(1, 3, figsize=(14, 4.4), gridspec_kw={"width_ratios": [1, 1.15, 1.0], "wspace": 0.55})
# A: mortality by first-GCS band, all / intubated / not intubated
a = ax[0]; w = 0.26
series = [("All", GRP[GRP.grouping == "gcs_band"], C["grey"]),
          ("Not intubated", GRP[(GRP.grouping == "intubated x gcs_band") & (GRP.intubated == 0)], C["model"]),
          ("Intubated", GRP[(GRP.grouping == "intubated x gcs_band") & (GRP.intubated == 1)], C["apache"])]
for i, (lab, g, col) in enumerate(series):
    g = g.set_index("gcs_band").reindex(BANDS)
    x = np.arange(3) + (i - 1) * w
    a.bar(x, g.mortality * 100, w, color=col, label=lab, alpha=0.9)
    a.errorbar(x, g.mortality * 100, yerr=[(g.mortality - g.lo).clip(lower=0) * 100,
               (g.hi - g.mortality).clip(lower=0) * 100], fmt="none", ecolor="black", lw=0.8, capsize=2)
    for xx, nn in zip(x, g.n):
        if pd.notna(nn):
            a.text(xx, 1, f"{int(nn)}", ha="center", va="bottom", fontsize=6, rotation=90, color="white")
a.set_xticks(range(3)); a.set_xticklabels(BANDS); a.set_xlabel("First GCS in ICU (0-24 h)")
a.set_ylabel("28-day mortality, % (95% CI)"); a.set_title("A  Mortality by first GCS", loc="left")
a.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3, frameon=False, fontsize=7, columnspacing=0.8)
# B: mortality by trajectory within baseline band
b = ax[1]
TRJ = ["Improving (>=+2)", "Stable (-1 to +1)", "Worsening (<=-2)"]
tcol = [C["lr"], C["grey"], C["pink"]]
gt = GRP[GRP.grouping == "gcs_band x trajectory"]
for i, (t, col) in enumerate(zip(TRJ, tcol)):
    g = gt[gt.trajectory == t].set_index("gcs_band").reindex(BANDS)
    x = np.arange(3) + (i - 1) * w
    ok = g.n.fillna(0) >= 20                       # do not draw bars from < 20 patients
    a_ = g.mortality.where(ok) * 100
    b.bar(x, a_, w, color=col, label=t, alpha=0.9)
    b.errorbar(x[ok.values], a_[ok].values,
               yerr=[((g.mortality - g.lo).clip(lower=0) * 100)[ok].values,
                     ((g.hi - g.mortality).clip(lower=0) * 100)[ok].values],
               fmt="none", ecolor="black", lw=0.8, capsize=2)
    for xx, nn, okk in zip(x, g.n, ok):
        if not okk:
            b.text(xx, 2, "n<20", ha="center", fontsize=6, rotation=90, color=C["grey"])
b.set_xticks(range(3)); b.set_xticklabels(BANDS); b.set_xlabel("First GCS in ICU (0-24 h)")
b.set_title("B  Mortality by GCS change over 24 h", loc="left")
b.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3, frameon=False, fontsize=7, columnspacing=0.8)
# C: OR of first GCS (per point) across models
c = ax[2]
fm = MOD[MOD.term == "gcs_first"].reset_index(drop=True)
yy = np.arange(len(fm))[::-1]
c.axvline(1, color="black", lw=0.8, ls="--")
c.errorbar(fm.OR, yy, xerr=[fm.OR - fm.OR_lo, fm.OR_hi - fm.OR], fmt="o", color=C["model"],
           ecolor=C["model"], ms=4, capsize=2)
SHORT = ["M1 first GCS only", "M2 + intubation", "M3 + GCS trajectory", "M4 + intubation + trajectory",
         "M5 fully adjusted", "Not intubated only", "Intubated only", "Era <=2017 only",
         "Era >2017 only"]
assert len(SHORT) == len(fm)
c.set_yticks(yy); c.set_yticklabels(SHORT, fontsize=7)
c.set_xlabel("OR of 28-day death per 1-point higher first GCS\n"
             "(OR < 1: higher GCS = lower risk, expected;  OR > 1: 'paradox')")
c.set_title("C  Is a higher GCS linked to death?", loc="left")
fig.text(0.01, -0.16, "Whole cohort, raw first-24-h values (complete cases for models). Intubated = invasive "
         "ventilation or verbal score 'No Response-ETT' within 24 h. Trajectory = last minus first GCS in 0-24 h; "
         "a GCS of 15 cannot improve (ceiling effect). M5 = M4 + age, metastatic cancer, SOFA without CNS.", fontsize=7, style="italic")
save(fig, "figS3_gcs_paradox")
plt.show()

# ------------------------------------------ print ------------------------------------------
pf = lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
print(f"Frozen files unchanged: {n_frozen}")
print(f"\nComparator study GCS mean {he_mean} SD {he_sd}: the largest SD possible on a 3-15 scale with "
      f"that mean is {sd_max:.2f} -> their reported SD is impossible for a total GCS.")
print(f"Our first GCS (n={int(ours['count'])}): mean {ours['mean']:.2f} SD {ours['std']:.2f}, "
      f"median {D.gcs_first.median():.0f} [{D.gcs_first.quantile(.25):.0f}-{D.gcs_first.quantile(.75):.0f}]; "
      f"missing {D.gcs_first.isna().sum()}")
print(f"Intubated in first 24 h: {D.intubated.sum()} ({D.intubated.mean():.1%}); "
      f"mortality intubated {D[D.intubated == 1].y.mean():.1%} vs not {D[D.intubated == 0].y.mean():.1%}")
print("\nMortality by first-GCS band (all / not intubated / intubated):")
for band in BANDS:
    parts = []
    for lab, g, _ in series:
        r = g.set_index("gcs_band").reindex([band]).iloc[0]
        parts.append(f"{lab} {r.mortality:.1%} (n={int(r.n) if pd.notna(r.n) else 0})"
                     if pd.notna(r.mortality) else f"{lab} -")
    print(f"  GCS {band:>5}: " + " | ".join(parts))
print(f"\nModels (complete cases n={len(CC)}, deaths {int(CC.y.sum())}): OR per 1-point higher first GCS")
for _, r in MOD[MOD.term == "gcs_first"].iterrows():
    print(f"  {r.model:<52} OR {r.OR:.3f} ({r.OR_lo:.3f}-{r.OR_hi:.3f}) p={pf(r.p)}  n={r.n}"
          + (f"  AUC {r.auc:.3f}" if pd.notna(r.auc) else ""))
print("\nOther terms in M5:")
for _, r in MOD[(MOD.model == MODELS[-1][0]) & (MOD.term != "gcs_first")].iterrows():
    print(f"  {r.term:<38} OR {r.OR:.3f} ({r.OR_lo:.3f}-{r.OR_hi:.3f}) p={pf(r.p)}")
print("\nLikelihood-ratio tests:")
for k, (s_, d_, p_) in LRT.items():
    print(f"  {k:<28} chi2={s_:.1f} df={int(d_)} p={pf(p_)}")
print(f"Interaction first GCS x intubation: p={pf(p_inter)}")
print(f"\nHe et al. hypothesis - GCS change (last-first) by baseline band (Kruskal-Wallis p={pf(kw.pvalue)}):")
for band in BANDS:
    g = D[D.gcs_band == band].gcs_change.dropna()
    print(f"  GCS {band:>5}: median change {g.median():+.0f} [IQR {g.quantile(.25):+.0f} to {g.quantile(.75):+.0f}], "
          f"worsening (<=-2) {(g <= -2).mean():.1%}  (n={len(g)})")
print("\nMortality by trajectory within baseline band:")
for _, r in gt.assign(_b=gt.gcs_band.map(BANDS.index), _t=gt.trajectory.map(TRJ.index)).sort_values(["_b", "_t"]).iterrows():
    print(f"  GCS {r.gcs_band:>5} | {r.trajectory:<18} {r.mortality:.1%} ({r.lo:.1%}-{r.hi:.1%})  n={r.n}")

# automatic verdict (check by eye)
m1 = MOD[(MOD.model == MODELS[0][0]) & (MOD.term == "gcs_first")].iloc[0]
m4 = MOD[(MOD.model == MODELS[3][0]) & (MOD.term == "gcs_first")].iloc[0]
m5 = MOD[(MOD.model == MODELS[4][0]) & (MOD.term == "gcs_first")].iloc[0]
print("\nVERDICT (automatic, check by eye):")
if m1.OR_lo > 1:
    print("  H0: the paradox REPLICATES unadjusted (higher first GCS -> higher mortality).")
elif m1.OR_hi < 1:
    print("  H0: NO paradox unadjusted - higher first GCS -> LOWER mortality (expected direction).")
else:
    print("  H0: no clear unadjusted association between first GCS and mortality (CI includes 1).")
print(f"  After intubation + trajectory (M4): OR {m4.OR:.3f} ({m4.OR_lo:.3f}-{m4.OR_hi:.3f}); "
      f"fully adjusted (M5): OR {m5.OR:.3f} ({m5.OR_lo:.3f}-{m5.OR_hi:.3f})")
print("\nSaved: nb4/gcs_paradox_models.csv, nb4/gcs_paradox_groups.csv, figures_v2/figS3_gcs_paradox")

In [ ]:
# ========= NB4 CELL 11: FIGURE 1 - study flow diagram (TRIPOD+AI item 20a) =======================
# Redrawn for the paper from the frozen NB1 counts (attrition_counts_final.csv) and the frozen split
# file, so every number comes from a file (nothing typed by hand). Adds what the NB1 diagram did not
# show: both validation designs, deaths per set, nested CV, and the intended-use test subgroup
# (CAM-ICU positive by 24 h). Every count is cross-checked against the other frozen files.
# Output: figures_v2/fig1_flow_diagram (.png 300 dpi + .pdf vector), nb4/fig1_flow_counts.csv
import importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import v2_nb4
importlib.reload(v2_nb4)
from v2_nb4 import P2, OUT4, freeze_check, style, save

n_frozen = freeze_check(); style()
ATT = pd.read_csv(P2 + "attrition_counts_final.csv")
CO = pd.read_csv(P2 + "cohort_final.csv")[["stay_id", "mortality_28d"]]
SP = pd.read_csv(P2 + "splits_v2.csv")
DT = pd.read_csv(P2 + "cohort_delirium_timing.csv")[["stay_id", "cam_positive_by_24h"]]
assert len(ATT) == 8 and list(ATT.columns) == ["stage", "n"], "unexpected attrition file layout"
n = ATT.n.astype(int).tolist()
D = CO.merge(SP, on="stay_id").merge(DT, on="stay_id")

# ------------------------------------- consistency checks -------------------------------------
assert n[7] == len(CO) == len(SP) == len(DT) == len(D), "final cohort size differs between files"
assert all(a >= b for a, b in zip([n[0], n[2], n[4], n[5], n[6]], [n[2], n[4], n[5], n[6], n[7]])), \
    "attrition counts are not decreasing"
assert n[3] >= n[4], "unique patients < patients with ICU stay"
for arm in ["random_split", "temporal_split"]:
    assert set(D[arm]) == {"train", "test"}, f"{arm}: labels must be train/test"


def cnt(mask):
    s = D[mask]
    return len(s), int(s.mortality_28d.sum()), s.mortality_28d.mean()


tot = cnt(D.stay_id.notna())
rtr, rte = cnt(D.random_split == "train"), cnt(D.random_split == "test")
etr, ete = cnt(D.temporal_split == "train"), cnt(D.temporal_split == "test")
rte24 = cnt((D.random_split == "test") & (D.cam_positive_by_24h == 1))
ete24 = cnt((D.temporal_split == "test") & (D.cam_positive_by_24h == 1))
assert rtr[0] + rte[0] == tot[0] and etr[0] + ete[0] == tot[0]
assert rtr[1] + rte[1] == tot[1] and etr[1] + ete[1] == tot[1]
# the prediction files must hold exactly the test patients shown in the diagram
for arm, t in [("random_split", rte), ("temporal_split", ete)]:
    T = pd.read_csv(P2 + f"test_predictions_{arm}.csv")
    assert len(T) == t[0] and int(T.y.sum()) == t[1], f"{arm}: diagram counts differ from predictions"
    assert set(T.stay_id) == set(D.stay_id[D[arm] == "test"]), f"{arm}: test patient IDs differ"
yr = SP.groupby("temporal_split").est_calendar_year.agg(["min", "max"])

COUNTS = pd.DataFrame([
    ("Cancer admissions", n[0], None), ("Cancer + delirium admissions", n[2], None),
    ("Cancer + delirium patients", n[3], None), ("First ICU stay, age >=18", n[4], None),
    ("Valid ICU discharge time", n[5], None), ("Alive at 24 h", n[6], None),
    ("Final cohort", tot[0], tot[1]), ("Random split - development", rtr[0], rtr[1]),
    ("Random split - test", rte[0], rte[1]), ("Random split - test, CAM-ICU+ by 24 h", rte24[0], rte24[1]),
    ("Temporal split - development (<=2017)", etr[0], etr[1]),
    ("Temporal split - test (>2017)", ete[0], ete[1]),
    ("Temporal split - test, CAM-ICU+ by 24 h", ete24[0], ete24[1])], columns=["box", "n", "deaths"]).astype({"deaths": "Int64"})
COUNTS.to_csv(OUT4 + "fig1_flow_counts.csv", index=False)

# --------------------------------------------- figure ---------------------------------------------
f = lambda x: f"{x:,}"
pc = lambda t: f"{f(t[0])}; deaths {f(t[1])} ({t[2] * 100:.1f}%)"
fig, ax = plt.subplots(figsize=(7.2, 9.6))          # ~ Springer double-column width (174 mm)
ax.set_xlim(0, 10); ax.set_ylim(0, 13.4); ax.axis("off")
FS = 7.5


def box(x, y, w, h, t, fc="white", bold=False, fs=FS):
    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.04,rounding_size=0.08",
                                fc=fc, ec="black", lw=0.8))
    ax.text(x, y, t, ha="center", va="center", fontsize=fs, fontweight="bold" if bold else "normal",
            linespacing=1.35)


def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", lw=0.8, color="black", mutation_scale=8))


MX, MW, MH = 3.3, 5.0, 0.78
EX, EW, EH = 8.1, 3.7, 0.72
main = [f"Admissions with ≥1 cancer ICD-9/10 code\n(2,052 codes; MIMIC-IV v3.1, 2008-2022)\nn = {f(n[0])} admissions",
        f"Cancer AND delirium (ICD F05/293.0/293.1\nor positive CAM-ICU*)\nn = {f(n[2])} admissions ({f(n[3])} patients)",
        f"First ICU stay, age ≥18 years\nn = {f(n[4])} patients",
        f"Valid ICU discharge time\nn = {f(n[5])}",
        f"Alive at 24 h after ICU admission\nn = {f(n[6])}",
        f"FINAL COHORT (in ICU at 24 h = prediction time)\nn = {f(tot[0])}; 28-day deaths {f(tot[1])} ({tot[2] * 100:.1f}%)"]
excl = [f"no delirium evidence\nn = {f(n[0] - n[2])} admissions",
        f"no qualifying ICU stay\n(or age <18 years)\nn = {f(n[3] - n[4])} patients",
        f"missing ICU\ndischarge time\nn = {f(n[4] - n[5])}",
        f"died within 24 h\nof ICU admission\nn = {f(n[5] - n[6])}",
        f"discharged alive\nfrom ICU within 24 h\nn = {f(n[6] - n[7])}"]
ys = [12.85, 11.55, 10.3, 9.15, 8.0, 6.75]
for i, (y, t) in enumerate(zip(ys, main)):
    last = i == len(main) - 1
    box(MX, y, MW + (0.6 if last else 0), MH, t, fc="#EAF2FB" if last else "white", bold=last)
    if i:
        arrow(MX, ys[i - 1] - MH / 2, MX, y + MH / 2)
        ym = (ys[i - 1] + y) / 2
        ax.plot([MX, EX - EW / 2], [ym, ym], color="black", lw=0.8)
        box(EX, ym, EW, EH, "Excluded: " + excl[i - 1], fc="#FBEDED", fs=7)

# two validation designs
ax.text(2.45, 5.75, "Analysis 1: random split (80/20, stratified)", ha="center", fontsize=FS, fontweight="bold")
ax.text(7.55, 5.75, "Analysis 2: temporal split (calendar era)", ha="center", fontsize=FS, fontweight="bold")
arrow(2.45, ys[-1] - MH / 2, 2.45, 5.95)
right_edge = MX + (MW + 0.6) / 2
ax.plot([right_edge, 7.55], [ys[-1], ys[-1]], color="black", lw=0.8)
arrow(7.55, ys[-1], 7.55, 5.95)
BW, BH = 4.3, 1.05
box(2.45, 4.95, BW, BH, f"Development set\nn = {pc(rtr)}\nPreparation, feature selection, tuning,\n"
    "nested 5×5 cross-validation, calibration")
box(7.55, 4.95, BW, BH, f"Development era ({yr.loc['train', 'min']}-{yr.loc['train', 'max']})\n"
    f"n = {pc(etr)}\nAll steps repeated on this era only")
arrow(2.45, 4.95 - BH / 2, 2.45, 3.55 + BH / 2); arrow(7.55, 4.95 - BH / 2, 7.55, 3.55 + BH / 2)
box(2.45, 3.55, BW, BH, f"Internal test set (held out, opened once)\nn = {pc(rte)}", fc="#EEF7EE")
box(7.55, 3.55, BW, BH, f"Temporal test set ({yr.loc['test', 'min']}-{yr.loc['test', 'max']}; opened once)\n"
    f"n = {pc(ete)}", fc="#EEF7EE")
arrow(2.45, 3.55 - BH / 2, 2.45, 2.2 + 0.7 / 2); arrow(7.55, 3.55 - BH / 2, 7.55, 2.2 + 0.7 / 2)
box(2.45, 2.2, BW, 0.7, f"Intended-use subgroup: CAM-ICU+ by 24 h\nn = {pc(rte24)}", fc="#FFF8E6")
box(7.55, 2.2, BW, 0.7, f"Intended-use subgroup: CAM-ICU+ by 24 h\nn = {pc(ete24)}", fc="#FFF8E6")
ax.text(5.0, 1.15, "*CAM-ICU positive = acute change AND inattention AND (disorganised thinking OR altered level of "
        "consciousness)\nat the same assessment. Outcome: death within 28 days of ICU admission. "
        "The two analyses are independent:\neach uses the whole final cohort, divided in a different way.",
        ha="center", va="top", fontsize=6.3, style="italic", linespacing=1.4)
save(fig, "fig1_flow_diagram")
plt.show()

print(f"Frozen files unchanged: {n_frozen}")
print("All flow-diagram counts cross-checked against cohort, split, delirium-timing and test-prediction files.")
print(COUNTS.to_string(index=False))
print("Saved: figures_v2/fig1_flow_diagram (.png/.pdf), nb4/fig1_flow_counts.csv")

In [ ]:
# ========= NB4 CELL 12: VERIFICATION PASS 1 - independent recomputation of every key number ======
# Angle: "do the saved numbers follow from the frozen predictions?"
# Every headline number in the paper is recomputed here from the frozen patient-level test
# predictions with NEW code written only for this check (own AUROC, own DeLong, own average
# precision, own Brier / threshold metrics / calibration via statsmodels, own tiers and uncertainty
# flags, own subgroup counts) - none of the helper functions used to make the results
# (v2_eval / v2_final) is imported. Each recomputed value is compared with what the results files
# and paper tables say.
# It also writes the PAPER NUMBER REGISTRY (nb4/verification/paper_numbers.csv): the single list of
# numbers (with CI and CI method) that the manuscript must use, so no number is copied by hand.
# Read-only on all results. Output: nb4/verification/V1_recompute.csv, nb4/verification/paper_numbers.csv
import os, json, re, hashlib, importlib, warnings
import numpy as np, pandas as pd
import statsmodels.api as sm
from scipy import stats
from scipy.stats import rankdata
import v2_nb4
importlib.reload(v2_nb4)
rd = lambda *a, **k: pd.read_csv(*a, float_precision="round_trip", **k)   # exact floats (no last-digit parsing error)
from v2_nb4 import P2, M2, OUT4, ARMS, freeze_check

warnings.filterwarnings("ignore")
VER = OUT4 + "verification/"
os.makedirs(VER, exist_ok=True)
R = []


def check(section, name, ok, detail=""):
    R.append({"section": section, "check": name, "result": "PASS" if bool(ok) else "FAIL",
              "detail": str(detail)[:300]})


close = lambda a, b, tol=1e-9: (pd.isna(a) and pd.isna(b)) or (
    not pd.isna(a) and not pd.isna(b) and abs(float(a) - float(b)) <= tol)


# ----------------------------- independent implementations -----------------------------
def my_auc(y, p):
    y = np.asarray(y).astype(int); r = rankdata(p); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def my_delong(y, preds):
    """Sun & Xu (2014) fast DeLong; returns aucs and covariance matrix."""
    y = np.asarray(y).astype(int); preds = np.atleast_2d(preds)
    pos, neg = preds[:, y == 1], preds[:, y == 0]
    m, n = pos.shape[1], neg.shape[1]
    aucs, v01, v10 = [], [], []
    for k in range(preds.shape[0]):
        allr = rankdata(np.r_[pos[k], neg[k]])
        rp, rn = rankdata(pos[k]), rankdata(neg[k])
        aucs.append((allr[:m].sum() - m * (m + 1) / 2) / (m * n))
        v01.append((allr[:m] - rp) / n)              # placement of each positive
        v10.append(1 - (allr[m:] - rn) / m)          # placement of each negative
    v01, v10 = np.array(v01), np.array(v10)
    cov = np.atleast_2d(np.cov(v01)) / m + np.atleast_2d(np.cov(v10)) / n
    return np.array(aucs), cov


def my_delong_ci(y, p):
    a, c = my_delong(y, p)
    se = np.sqrt(c[0, 0])
    return a[0], a[0] - 1.96 * se, a[0] + 1.96 * se


def my_delong_p(y, p1, p2):
    a, c = my_delong(y, np.vstack([p1, p2]))
    z = (a[0] - a[1]) / np.sqrt(c[0, 0] + c[1, 1] - 2 * c[0, 1])
    return 2 * stats.norm.sf(abs(z))


def my_ap(y, p):
    y = np.asarray(y).astype(int); p = np.asarray(p)
    o = np.argsort(-p, kind="mergesort"); y, p = y[o], p[o]
    last = np.r_[np.where(np.diff(p))[0], len(p) - 1]      # last index of each distinct score
    tp = np.cumsum(y)[last]; fp = (last + 1) - tp
    prec, rec = tp / (tp + fp), tp / y.sum()
    return float(np.sum(np.diff(np.r_[0, rec]) * prec))


def my_thr(y, p, t):
    y = np.asarray(y).astype(int); pr = np.asarray(p) >= t
    tp, fp = int((pr & (y == 1)).sum()), int((pr & (y == 0)).sum())
    tn, fn = int((~pr & (y == 0)).sum()), int((~pr & (y == 1)).sum())
    dv = lambda a, b: a / b if b else np.nan
    return dict(tp=tp, fp=fp, tn=tn, fn=fn, sensitivity=dv(tp, tp + fn), specificity=dv(tn, tn + fp),
                ppv=dv(tp, tp + fp), npv=dv(tn, tn + fn))


def my_cal(y, p):
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    if np.ptp(lp) == 0:                              # constant risk: slope undefined
        return np.nan, np.nan
    slope = sm.GLM(y, np.column_stack([np.ones(len(lp)), lp]), family=sm.families.Binomial()).fit().params[1]
    inter = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(), offset=lp).fit().params[0]
    return float(slope), float(inter)


def my_nb(y, p, t):
    y = np.asarray(y); pr = np.asarray(p) >= t; n = len(y)
    return ((pr & (y == 1)).sum() - (pr & (y == 0)).sum() * t / (1 - t)) / n


fp12 = lambda ids: hashlib.md5(",".join(map(str, sorted(ids))).encode()).hexdigest()[:12]
FP = {"random_split": "0a23991d2383", "temporal_split": "8a4eb81d4c96"}

# ------------------------------------------- inputs -------------------------------------------
n_frozen = freeze_check()
check("0 setup", "frozen NB1-NB3 files unchanged (md5 manifest)", True, f"{n_frozen} files")
CO = rd(P2 + "cohort_final.csv").set_index("stay_id")
SP = rd(P2 + "splits_v2.csv").set_index("stay_id")
DT = rd(P2 + "cohort_delirium_timing.csv").set_index("stay_id")
MT = rd(P2 + "model_table_v2.csv", usecols=["stay_id", "cancer_type", "sex_male",
                                                       "age_at_admission"]).set_index("stay_id")
H = rd(P2 + "test_headline_metrics.csv").set_index("arm")
SG = rd(P2 + "test_subgroups.csv")
T3 = rd(OUT4 + "table3_risk_tiers.csv")
T6 = rd(OUT4 + "table6_delirium24h_deployment.csv")
CAL = rd(OUT4 + "calibration_metrics.csv")
DCA = rd(OUT4 + "decision_curve_all.csv")
T2 = rd(OUT4 + "tables/Table2_models.csv").set_index("Model")
PH = rd(OUT4 + "posthoc_model_comparisons.csv")
CV = rd(P2 + "cv/cv_summary.csv").query("pool == 'B_primary'").set_index("model")
T2NAME = {"SoftVoting_calibrated": "Soft-Voting ensemble, isotonic (FINAL)",
          "SoftVoting_VennAbers": "Soft-Voting ensemble, Venn-Abers", "Stacking": "Stacking ensemble",
          "XGB": "XGBoost", "LightGBM": "LightGBM", "CatBoost": "CatBoost", "RF": "Random forest",
          "LR": "Logistic regression", "DT": "Decision tree", "MultiView_meta": "Multi-View (diagnostic)",
          "SOFA": "SOFA score", "APACHE-II": "APACHE II score"}
REG = []   # paper number registry


def reg(item, arm, value, lo=np.nan, hi=np.nan, method="", source="", digits=3):
    disp = f"{value:.{digits}f}" if pd.isna(lo) else f"{value:.{digits}f} ({lo:.{digits}f}-{hi:.{digits}f})"
    REG.append({"item": item, "arm": arm, "value": value, "ci_lo": lo, "ci_hi": hi,
                "ci_method": method, "display": disp, "source": source})


# ------------------------------------ cohort-level checks ------------------------------------
y_all = CO.mortality_28d
check("1 cohort", "final cohort 2,777 patients / 1,060 deaths", len(CO) == 2777 and y_all.sum() == 1060,
      f"{len(CO)} / {int(y_all.sum())}")
check("1 cohort", "split file covers exactly the cohort", set(SP.index) == set(CO.index))
reg("Final cohort, n", "all", len(CO), digits=0, source="cohort_final.csv")
reg("Final cohort, 28-day deaths", "all", int(y_all.sum()), digits=0, source="cohort_final.csv")
reg("Final cohort, 28-day mortality %", "all", 100 * y_all.mean(), digits=1, source="cohort_final.csv")
T1 = rd(OUT4 + "table1_baseline.csv")
row = T1[T1.Characteristic.str.startswith("Patients")].iloc[0]
check("1 cohort", "Table 1 patient counts = cohort (overall / survived / died)",
      [int(str(row[k]).replace(",", "")) for k in ["Overall", "Survived", "Died within 28 days"]] ==
      [len(CO), int((y_all == 0).sum()), int(y_all.sum())], f"{row.Overall}/{row.Survived}/{row['Died within 28 days']}")
F1 = rd(OUT4 + "fig1_flow_counts.csv").set_index("box")
check("1 cohort", "Figure 1 final-cohort box = cohort", int(F1.loc["Final cohort", "n"]) == len(CO))

# --------------------------------------- per-arm checks ---------------------------------------
for arm in ARMS:
    S = f"2 {arm}"
    T = rd(P2 + f"test_predictions_{arm}.csv")
    L = json.load(open(M2 + f"locked_rules_{arm}.json"))
    A = rd(P2 + f"test_model_auc_{arm}.csv").set_index("model")
    DL = rd(P2 + f"test_delong_{arm}.csv").set_index("comparison")
    NB = rd(P2 + f"test_net_benefit_{arm}.csv")
    y = T.y.values.astype(int); pc = T.p_SoftVoting_calibrated.values; h = H.loc[arm]
    test_ids = SP.index[SP[arm] == "test"]
    tag = "random" if arm == "random_split" else "temporal"

    # identity of the test set
    check(S, "test patients = split file test patients", set(T.stay_id) == set(test_ids), len(T))
    check(S, "test fingerprint equals the one locked in NB3", fp12(T.stay_id) == FP[arm], fp12(T.stay_id))
    check(S, "outcome in predictions = cohort outcome", (CO.loc[T.stay_id, "mortality_28d"].values == y).all())
    pcols = [c for c in T.columns if c.startswith("p_")] + ["va_p", "va_p0", "va_p1"]
    check(S, "all probabilities present and inside [0,1]",
          T[pcols].notna().all().all() and ((T[pcols] >= 0) & (T[pcols] <= 1)).all().all())
    check(S, "Venn-Abers p0 <= p1 and width = p1 - p0",
          (T.va_p0 <= T.va_p1 + 1e-12).all() and np.allclose(T.va_width, T.va_p1 - T.va_p0))

    # AUROC + DeLong CI of every model
    bad = []
    for model in A.index:
        col = "va_p" if model == "SoftVoting_VennAbers" else "p_" + model
        a, lo, hi = my_delong_ci(y, T[col].values)
        if not (close(a, A.loc[model, "auc"], 1e-9) and close(lo, A.loc[model, "lo"], 1e-6)
                and close(hi, A.loc[model, "hi"], 1e-6)):
            bad.append(f"{model} (diff AUROC {a - A.loc[model, 'auc']:.2e}, lo {lo - A.loc[model, 'lo']:.2e}, "
                       f"hi {hi - A.loc[model, 'hi']:.2e})")
        if not pd.isna(A.loc[model, "brier"]) and not close(np.mean((T[col] - y) ** 2), A.loc[model, "brier"], 1e-9):
            bad.append(model + " Brier")
    check(S, f"AUROC + DeLong CI of all {len(A)} models recomputed", not bad, bad)
    # Table 2 strings + AUPRC + Brier
    bad = []
    for model, name in T2NAME.items():
        col = "va_p" if model == "SoftVoting_VennAbers" else "p_" + model
        a, lo, hi = my_delong_ci(y, T[col].values)
        if T2.loc[name, f"AUROC {tag} (95% CI)"] != f"{a:.3f} ({lo:.3f}-{hi:.3f})":
            bad.append(name + " AUROC")
        if not close(round(my_ap(y, T[col].values), 3), float(T2.loc[name, f"AUPRC {tag}"]), 1e-9):
            bad.append(name + " AUPRC")
        br = T2.loc[name, f"Brier {tag}"]
        if br not in ("n/a*", "-") and not close(round(np.mean((T[col] - y) ** 2), 3), float(br), 1e-9):
            bad.append(name + " Brier")
    check(S, "Table 2 AUROC/AUPRC/Brier strings match recomputation", not bad, bad)
    # nested CV column of Table 2
    bad = [k for k, cvk in [("SoftVoting_calibrated", "SoftVoting"), ("Stacking", "Stacking"), ("LR", "LR"),
                            ("SOFA", "SOFA"), ("APACHE-II", "APACHE-II")]
           if T2.loc[T2NAME[k], "Nested CV AUROC, mean (SD)"] != f"{CV.loc[cvk, 'auc_mean']:.3f} ({CV.loc[cvk, 'auc_sd']:.3f})"]
    check(S, "Table 2 nested-CV column = cv_summary.csv", not bad, bad)

    # headline metrics
    tm = my_thr(y, pc, L["threshold"])
    slope, inter = my_cal(y, pc)
    ok = (close(h.auc, my_auc(y, pc)) and close(h.auprc, my_ap(y, pc)) and
          close(h.brier, np.mean((pc - y) ** 2)) and close(h.threshold, L["threshold"], 1e-12) and
          all(int(h[k]) == tm[k] for k in ["tp", "fp", "tn", "fn"]) and
          all(close(h[k], tm[k]) for k in ["sensitivity", "specificity", "ppv", "npv"]) and
          close(h.expected_observed_ratio, pc.mean() / y.mean()) and int(h.n) == len(y) and int(h.deaths) == y.sum())
    check(S, "headline AUROC/AUPRC/Brier/threshold metrics/E:O recomputed", ok)
    check(S, "calibration slope and intercept (statsmodels GLM vs original code)",
          close(slope, h.cal_slope, 2e-3) and close(inter, h.cal_intercept, 2e-3),
          f"slope {slope:.4f} vs {h.cal_slope:.4f}; intercept {inter:.4f} vs {h.cal_intercept:.4f}")
    c = CAL[(CAL.arm == arm) & (CAL.model == "Final model (Soft-Voting)")].iloc[0]
    check(S, "calibration_metrics.csv (Fig 3) final-model slope/Brier = headline",
          close(c.cal_slope, h.cal_slope, 2e-3) and close(c.brier, h.brier, 1e-9))

    # tiers + uncertainty flags from the locked rules
    tier = np.select([pc < L["tier_cut_low"], pc < L["tier_cut_high"]], ["Low", "Intermediate"], "High")
    check(S, "risk tier of every patient = locked rule applied to calibrated risk", (tier == T.tier.values).all())
    unc = np.zeros(len(y), bool)
    for cut in (L["tier_cut_low"], L["tier_cut_high"]):
        unc |= (T.va_p0.values < cut) & (T.va_p1.values >= cut)
    check(S, "'uncertain' flags = Venn-Abers interval crossing a tier cut-off",
          (unc == T.uncertain_tier.astype(bool).values).all())
    check(S, "'uncertain at threshold' flags recomputed",
          (((T.va_p0 < L["threshold"]) & (T.va_p1 >= L["threshold"])).values ==
           T.uncertain_threshold.astype(bool).values).all())
    bad = []
    for t in ["Low", "Intermediate", "High"]:
        r3 = T3[(T3.arm == arm) & (T3.tier == t)].iloc[0]; m = tier == t
        if int(r3.n) != m.sum() or int(r3.deaths) != y[m].sum():
            bad.append(t)
        if m.any() and not close(r3.pct_uncertain, 100 * unc[m].mean(), 1e-9):
            bad.append(t + " %uncertain")
    check(S, "Table 3 tier n / deaths / % uncertain recomputed", not bad, bad)

    # net benefit
    bad = [t for t, v in zip(NB.threshold, NB.model) if not close(v, my_nb(y, pc, t), 1e-12)]
    dca = DCA[(DCA.arm == arm) & (DCA.model == "Final model (Soft-Voting)")]
    bad += [f"dca {t}" for t, v in zip(dca.threshold, dca.net_benefit) if not close(v, my_nb(y, pc, t), 1e-12)]
    check(S, "net benefit (NB3 file + decision-curve file) recomputed", not bad, bad[:5])

    # pre-specified DeLong p-values (uncalibrated Soft-Voting)
    bad = []
    for comp, r in DL.iterrows():
        other = comp.replace("SoftVoting vs ", "")
        if not close(my_delong_p(y, T.p_SoftVoting.values, T["p_" + other].values), r.p_delong, 1e-6):
            bad.append(comp)
    check(S, "pre-specified DeLong p-values recomputed (own DeLong)", not bad, bad)

    # subgroups
    g = pd.DataFrame({"y": y, "p": pc,
                      "cancer_type": MT.loc[T.stay_id, "cancer_type"].values,
                      "sex": np.where(MT.loc[T.stay_id, "sex_male"].values == 1, "male", "female"),
                      "age_group": np.select([MT.loc[T.stay_id, "age_at_admission"].values < 60,
                                              MT.loc[T.stay_id, "age_at_admission"].values < 75],
                                             ["<60", "60-74"], ">=75"),
                      "delirium_by_24h": np.where(DT.loc[T.stay_id, "cam_positive_by_24h"].values == 1,
                                                  "CAM-ICU+ by 24h", "not by 24h")})
    bad = []
    for _, r in SG[SG.arm == arm].iterrows():
        d = g[g[r.subgroup].astype(str) == str(r.level)]
        if len(d) != r.n or d.y.sum() != r.deaths or not close(d.p.mean(), r.mean_predicted, 1e-9):
            bad.append(f"{r.subgroup}={r.level}")
        elif not pd.isna(r.auc) and not close(my_auc(d.y, d.p), r.auc, 1e-9):
            bad.append(f"{r.subgroup}={r.level} AUC")
    check(S, "subgroup n / deaths / mean risk / AUROC recomputed", not bad, bad)

    # Table 6 (intended-use population)
    cam = DT.loc[T.stay_id, "cam_positive_by_24h"].values == 1
    bad = []
    for grp, m in [("CAM-ICU positive by 24 h", cam), ("Not positive by 24 h", ~cam),
                   ("All test patients", np.ones(len(y), bool))]:
        r6 = T6[(T6.arm == arm) & (T6.group == grp)].iloc[0]
        tt = my_thr(y[m], pc[m], L["threshold"])
        if (int(r6.n) != m.sum() or int(r6.deaths) != y[m].sum() or not close(r6.auc_final, my_auc(y[m], pc[m]))
                or not close(r6.sensitivity, tt["sensitivity"]) or not close(r6.specificity, tt["specificity"])):
            bad.append(grp)
        for other in ["SOFA", "APACHE-II", "Stacking", "LR"]:
            if not close(r6[f"p_vs_{other}"], my_delong_p(y[m], T.p_SoftVoting.values[m], T["p_" + other].values[m]), 1e-6):
                bad.append(f"{grp} p vs {other}")
        if not close(r6["net_benefit_at_0.30"], my_nb(y[m], pc[m], 0.30), 1e-12):
            bad.append(grp + " NB")
    check(S, "Table 6 (CAM-ICU+ by 24 h) counts / AUROC / sens / spec / DeLong / NB recomputed", not bad, bad)

    # post-hoc comparisons (cell 9)
    bad = []
    for _, r in PH[PH.arm == arm].iterrows():
        m = cam if r.population.startswith("CAM") else np.ones(len(y), bool)
        c1 = "p_SoftVoting_calibrated" if "(calibrated" in r.comparison else "p_SoftVoting"
        c2 = "p_LR" if "logistic" in r.comparison else "p_MultiView_meta"
        if not (close(r.auc_final, round(my_auc(y[m], T[c1].values[m]), 4), 1e-9) and
                close(r.auc_comparator, round(my_auc(y[m], T[c2].values[m]), 4), 1e-9) and
                close(r.p_delong, my_delong_p(y[m], T[c1].values[m], T[c2].values[m]), 1e-6)):
            bad.append(r.comparison + " / " + r.population)
    check(S, "post-hoc comparisons (cell 9) recomputed", not bad, bad)

    # ------------------------------- registry for this arm -------------------------------
    src = f"test_predictions_{arm}.csv"
    a, lo, hi = my_delong_ci(y, pc)
    reg("Test set n", arm, len(y), digits=0, source=src)
    reg("Test set deaths", arm, int(y.sum()), digits=0, source=src)
    reg("Final model AUROC", arm, a, lo, hi, "DeLong", src)
    reg("Final model AUPRC", arm, h.auprc, h.auprc_lo, h.auprc_hi, "bootstrap 2,000 (stratified)", "test_headline_metrics.csv")
    reg("Final model Brier score", arm, h.brier, h.brier_lo, h.brier_hi, "bootstrap 2,000 (stratified)", "test_headline_metrics.csv")
    reg("Calibration slope", arm, h.cal_slope, source="test_headline_metrics.csv", digits=2)
    reg("Calibration intercept (calibration-in-the-large)", arm, h.cal_intercept, source="test_headline_metrics.csv", digits=2)
    reg("Expected/observed ratio", arm, h.expected_observed_ratio, source="test_headline_metrics.csv", digits=2)
    reg("ICI", arm, c.ICI, source="calibration_metrics.csv")
    reg("E90", arm, c.E90, source="calibration_metrics.csv")
    reg("Youden threshold (training OOF)", arm, L["threshold"], source=f"locked_rules_{arm}.json")
    for k in ["sensitivity", "specificity", "ppv", "npv"]:
        reg(f"{k.upper() if len(k) == 3 else k.capitalize()} at threshold", arm, h[k], h[f"{k}_lo"],
            h[f"{k}_hi"], "bootstrap 1,000 (stratified)", "test_headline_metrics.csv")
    for mdl, nm in [("Stacking", "Stacking"), ("LR", "Logistic regression"), ("SOFA", "SOFA"),
                    ("APACHE-II", "APACHE II"), ("XGB", "XGBoost"), ("MultiView_meta", "Multi-View")]:
        a2, l2, h2 = my_delong_ci(y, T["p_" + mdl].values)
        reg(f"{nm} AUROC", arm, a2, l2, h2, "DeLong", src)
    for comp, r in DL.iterrows():
        reg(f"DeLong p: {comp} (pre-specified)", arm, r.p_delong, source=f"test_delong_{arm}.csv", digits=4)
    reg("Tier cut-off Low/Intermediate", arm, L["tier_cut_low"], source=f"locked_rules_{arm}.json")
    reg("Tier cut-off Intermediate/High", arm, L["tier_cut_high"], source=f"locked_rules_{arm}.json")
    for t in ["Low", "Intermediate", "High"]:
        m = tier == t
        reg(f"Observed mortality % - {t} tier", arm, 100 * y[m].mean(), digits=1, source=src)
    reg("% uncertain (tier)", arm, 100 * unc.mean(), digits=1, source=src)
    for grp in ["CAM-ICU positive by 24 h", "Not positive by 24 h"]:
        r6 = T6[(T6.arm == arm) & (T6.group == grp)].iloc[0]
        reg(f"AUROC - {grp}", arm, r6.auc_final, r6.auc_lo, r6.auc_hi, "bootstrap 2,000 (stratified)",
            "table6_delirium24h_deployment.csv")
        reg(f"n - {grp}", arm, r6.n, digits=0, source="table6_delirium24h_deployment.csv")
    nb30 = my_nb(y, pc, 0.30)
    reg("Net benefit at 30% threshold (final model)", arm, nb30, source=src)
    reg("Net benefit at 30% threshold (treat all)", arm, y.mean() - (1 - y.mean()) * 0.3 / 0.7, source=src)

# ----------------------------- cross-validation, sample size, GCS -----------------------------
for mdl in ["SoftVoting", "Stacking", "XGB", "LR", "SOFA", "APACHE-II"]:
    reg(f"Nested CV AUROC (mean, SD in ci_lo) - {mdl}", "random_split", CV.loc[mdl, "auc_mean"],
        CV.loc[mdl, "auc_sd"], np.nan, "SD over 25 folds", "cv/cv_summary.csv")
RI = rd(OUT4 + "sample_size_riley.csv")
for _, r in RI.iterrows():
    S_ = 0.9
    n1 = r.parameters_P / ((S_ - 1) * np.log(1 - r.R2_CoxSnell / S_))
    n3 = (1.96 / 0.05) ** 2 * r.prevalence * (1 - r.prevalence)
    check("3 sample size", f"{r.arm}: Riley criteria 1 and 3 recomputed (from rounded R2)",
          abs(n1 - r.n_crit1_shrinkage) <= 5 and abs(n3 - r.n_crit3_intercept) <= 2,
          f"crit1 {n1:.0f} vs {r.n_crit1_shrinkage}; crit3 {n3:.0f} vs {r.n_crit3_intercept}")
    check("3 sample size", f"{r.arm}: n_train and events match the split",
          r.n_train == (SP[r.arm] == "train").sum() and
          r.events == int(CO.loc[SP.index[SP[r.arm] == "train"], "mortality_28d"].sum()))
    reg("Riley minimum sample size", r.arm, r.n_required, digits=0, source="sample_size_riley.csv")
    reg("Events per predictor parameter", r.arm, r.EPP, digits=1, source="sample_size_riley.csv")

GM = rd(OUT4 + "gcs_paradox_models.csv")
MT2 = rd(P2 + "model_table_v2.csv", usecols=["stay_id", "GCS_Total_first", "GCS_Total_slope",
                  "GCS_Total_late_minus_early_mean", "age_at_admission", "elix_metacanc", "sofa_total", "sofa_cns"])
d = MT2.merge(CO[["mortality_28d"]], left_on="stay_id", right_index=True).dropna()
f1 = sm.GLM(d.mortality_28d, sm.add_constant(d[["GCS_Total_first"]]), family=sm.families.Binomial()).fit()
g1 = GM[(GM.model.str.startswith("M1")) & (GM.term == "gcs_first")].iloc[0]
check("4 GCS paradox", "M1 odds ratio refitted with GLM on the same complete cases",
      abs(np.exp(f1.params["GCS_Total_first"]) - g1.OR) < 1e-4 and len(d) == g1.n,
      f"{np.exp(f1.params['GCS_Total_first']):.4f} vs {g1.OR:.4f}; n {len(d)} vs {g1.n}")
for mname in ["M1", "M2", "M3", "M4", "M5"]:
    r = GM[(GM.model.str.startswith(mname)) & (GM.term == "gcs_first")].iloc[0]
    reg(f"GCS paradox OR per point - {r.model}", "all", r.OR, r.OR_lo, r.OR_hi, "Wald", "gcs_paradox_models.csv")
for _, r in PH.iterrows():
    reg(f"Post-hoc: {r.comparison} [{r.population}] AUROC difference", r.arm, r["diff"], r.diff_lo, r.diff_hi,
        f"paired bootstrap 2,000; DeLong p={r.p_delong:.4f}; Holm p={r.p_holm_within_arm:.4f}",
        "posthoc_model_comparisons.csv", digits=3)

# ------------------------------------------- report -------------------------------------------
REP = pd.DataFrame(R)
REP.to_csv(VER + "V1_recompute.csv", index=False)
REGD = pd.DataFrame(REG)
REGD.to_csv(VER + "paper_numbers.csv", index=False)
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110)
print(f"VERIFICATION PASS 1 - independent recomputation: {(REP.result == 'PASS').sum()} PASS, "
      f"{(REP.result == 'FAIL').sum()} FAIL\n")
print(REP[["section", "check", "result"]].to_string(index=False))
if (REP.result == "FAIL").any():
    print("\nFAILED CHECKS (details):")
    print(REP[REP.result == "FAIL"][["section", "check", "detail"]].to_string(index=False))
print(f"\nPaper number registry: {len(REGD)} numbers -> nb4/verification/paper_numbers.csv")
print("Saved: nb4/verification/V1_recompute.csv")

In [ ]:
# ========= NB4 CELL 13: VERIFICATION PASS 2 - leakage, timeline, cross-table consistency, privacy ==
# Angle: "was the method followed, do the tables agree with each other, and is it safe to share?"
#   A  NB3 verification still has no FAIL + frozen files unchanged
#   B  Leakage: train/test separation (patients AND people), era boundary, out-of-fold file only
#      holds training patients, prepared files match the splits, no outcome-like or
#      measurement-process feature in the final models, final features exist in the prepared data
#   C  Locked rules reproduce from the TRAINING out-of-fold predictions (tertiles, Youden)
#   D  Timeline: every model / rule / selection file was last written BEFORE the test set was first
#      opened (models_v2/test_opened_log.json); test results were written after
#   E  Cross-table consistency: S2, S3, S5, S6, S7, S9, Table 3, Table 6, heterogeneity, Riley
#      against their source files
#   F  Figures: every paper figure exists as 300-dpi PNG + vector PDF
#   G  Privacy: which NB4 files are patient-level (must never be shared or pushed to GitHub)
# Read-only. Output: nb4/verification/V2_audit.csv, nb4/verification/patient_level_files_DO_NOT_SHARE.csv
import os, glob, json, re, datetime, importlib, warnings
import numpy as np, pandas as pd, joblib
from scipy import stats
import v2_nb4, v2_select
for m in (v2_nb4, v2_select):
    importlib.reload(m)
from v2_nb4 import P2, M2, FIG, OUT4, ARMS, freeze_check
from v2_select import is_measurement
rd = lambda *a, **k: pd.read_csv(*a, float_precision="round_trip", **k)   # exact floats (no last-digit parsing error)

warnings.filterwarnings("ignore")
VER = OUT4 + "verification/"
os.makedirs(VER, exist_ok=True)
R = []


def check(section, name, ok, detail=""):
    R.append({"section": section, "check": name, "result": "PASS" if bool(ok) else "FAIL",
              "detail": str(detail)[:300]})


# ------------------------------------------- A -------------------------------------------
n_frozen = freeze_check()
check("A frozen", "NB1-NB3 outputs unchanged since NB4 started", True, f"{n_frozen} files")
V3 = rd(P2 + "nb3_verification.csv")
check("A frozen", "NB3 verification file: no FAIL (INFO rows are numbers for the paper, not checks)",
      (V3.result == "FAIL").sum() == 0 and set(V3.result) <= {"PASS", "INFO"},
      f"{(V3.result == 'PASS').sum()} PASS, {(V3.result == 'FAIL').sum()} FAIL, {(V3.result == 'INFO').sum()} INFO")

# ------------------------------------------- B -------------------------------------------
CO = rd(P2 + "cohort_final.csv").set_index("stay_id")
SP = rd(P2 + "splits_v2.csv").set_index("stay_id")
check("B leakage", "one ICU stay per person (subject_id unique)", CO.subject_id.is_unique)
LEAK = re.compile(r"mortal|death|dod|expire|outtime|discharge|los_|_los|dischtime|deathtime|hospice|dnr|comfort",
                  re.I)
for arm in ARMS:
    tr, te = SP.index[SP[arm] == "train"], SP.index[SP[arm] == "test"]
    check("B leakage", f"{arm}: no ICU stay in both training and test", not set(tr) & set(te))
    check("B leakage", f"{arm}: no person in both training and test",
          not set(CO.loc[tr, "subject_id"]) & set(CO.loc[te, "subject_id"]))
    check("B leakage", f"{arm}: training + test = whole cohort", len(tr) + len(te) == len(CO))
    if arm == "temporal_split":
        check("B leakage", "temporal split: all training <= 2017 < all test",
              SP.loc[tr, "est_calendar_year"].max() <= 2017 < SP.loc[te, "est_calendar_year"].min(),
              f"train max {SP.loc[tr, 'est_calendar_year'].max()}, test min {SP.loc[te, 'est_calendar_year'].min()}")
    else:
        d = abs(CO.loc[tr, "mortality_28d"].mean() - CO.loc[te, "mortality_28d"].mean())
        check("B leakage", "random split: stratified (mortality differs < 1 point)", d < 0.01, f"{d:.4f}")
    oof = rd(P2 + f"train_oof_final_{arm}.csv")
    check("B leakage", f"{arm}: out-of-fold file = training patients only", set(oof.stay_id) == set(tr))
    ptr = rd(P2 + f"prep_{arm}_train.csv", usecols=["stay_id"])
    pte = rd(P2 + f"prep_{arm}_test.csv", nrows=0).columns
    pte_ids = rd(P2 + f"prep_{arm}_test.csv", usecols=["stay_id"])
    ptr_cols = rd(P2 + f"prep_{arm}_train.csv", nrows=0).columns
    check("B leakage", f"{arm}: prepared train/test files hold the split's patients",
          set(ptr.stay_id) == set(tr) and set(pte_ids.stay_id) == set(te))
    check("B leakage", f"{arm}: prepared train and test have identical columns", list(ptr_cols) == list(pte))
    B = joblib.load(M2 + f"final_{arm}.joblib")
    feats = list(B["features"])
    sel = rd(P2 + f"selected_features_{arm}.csv").feature.tolist()
    check("B leakage", f"{arm}: final model features = selected_features file", feats == sel or set(feats) == set(sel),
          f"{len(feats)} features")
    check("B leakage", f"{arm}: no outcome-like name among final features",
          not [f for f in feats if LEAK.search(f)], [f for f in feats if LEAK.search(f)])
    check("B leakage", f"{arm}: no measurement-process feature (pool B rule)",
          not [f for f in feats if is_measurement(f)], [f for f in feats if is_measurement(f)])
    check("B leakage", f"{arm}: every final feature exists in the prepared data",
          set(feats) <= set(ptr_cols), sorted(set(feats) - set(ptr_cols))[:5])
    check("B leakage", f"{arm}: outcome not among prepared predictors", "mortality_28d" not in feats)

    # ------------------------------------------- C -------------------------------------------
    L = json.load(open(M2 + f"locked_rules_{arm}.json"))
    q1, q2 = np.quantile(oof.oof_calibrated, [1 / 3, 2 / 3])
    check("C locked rules", f"{arm}: tier cut-offs = tertiles of training OOF risk",
          np.isclose(q1, L["tier_cut_low"]) and np.isclose(q2, L["tier_cut_high"]))
    yv, pv = oof.y.values, oof.oof_calibrated.values
    J = lambda t: (pv[yv == 1] >= t).mean() + (pv[yv == 0] < t).mean() - 1
    jmax = max(J(t) for t in np.unique(pv))
    check("C locked rules", f"{arm}: decision threshold maximises Youden J on training OOF",
          np.isclose(J(L["threshold"]), jmax, atol=1e-12), f"J(thr) {J(L['threshold']):.5f} vs max {jmax:.5f}")
    check("C locked rules", f"{arm}: threshold in model bundle = locked threshold",
          np.isclose(B["threshold"], L["threshold"]))

# ------------------------------------------- D -------------------------------------------
opened = datetime.datetime.fromisoformat(json.load(open(M2 + "test_opened_log.json"))["first_opened"])
slack = datetime.timedelta(seconds=1)                       # the log time is stored to the second
mt = lambda f: datetime.datetime.fromtimestamp(os.path.getmtime(f))
BEFORE = (glob.glob(M2 + "final_*.joblib") + glob.glob(M2 + "prep_*.joblib") + glob.glob(M2 + "tuning_*.json")
          + glob.glob(M2 + "locked_rules_*.json") + glob.glob(M2 + "feature_pool_decision.json")
          + glob.glob(P2 + "selected_features_*.csv") + glob.glob(P2 + "train_oof_final_*.csv")
          + glob.glob(P2 + "cv/*.csv") + glob.glob(P2 + "pilot_labs/*.csv"))
late = [os.path.relpath(f, "..") for f in BEFORE if mt(f) > opened + slack]
check("D timeline", f"{len(BEFORE)} model/rule/selection/CV files written before the test was opened ({opened})",
      not late, late[:6])
AFTER = glob.glob(P2 + "test_predictions_*.csv") + [P2 + "test_headline_metrics.csv"]
early = [os.path.relpath(f, "..") for f in AFTER if mt(f) < opened - slack]
check("D timeline", "test results written after the test was opened", not early, early)

# ------------------------------------------- E -------------------------------------------
TAB = OUT4 + "tables/"
pf = lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
S2 = rd(TAB + "S2_comparisons.csv", dtype={"p": str})
tt = rd(P2 + "cv/cv_corrected_ttests.csv").set_index("comparison")
bad = []
for _, r in S2.iterrows():
    if r.setting.startswith("Nested"):
        src_p, src_d = tt.loc[r.comparison, "p_value"], tt.loc[r.comparison, "mean_auc_diff"]
    else:
        arm = "random_split" if r.setting.startswith("Random") else "temporal_split"
        dl = rd(P2 + f"test_delong_{arm}.csv").set_index("comparison")
        src_p, src_d = dl.loc[r.comparison, "p_delong"], round(dl.loc[r.comparison, "diff"], 4)
    if str(r.p) != pf(src_p) or not np.isclose(r.mean_auc_diff, src_d, atol=1e-4):
        bad.append(r.comparison)
check("E tables", "S2 comparisons = CV t-test file + test DeLong files", not bad, bad)
S3 = rd(TAB + "S3_ablations.csv").set_index("model")
bad = []
for arm in ARMS:
    A = rd(P2 + f"test_model_auc_{arm}.csv").set_index("model")
    tag = "random" if arm == "random_split" else "temporal"
    for col, key in [("primary", "SoftVoting"), ("no TPIFE", "SV_noTPIFE"), ("with measurement", "SV_with_measurement")]:
        if not np.isclose(S3.loc["SoftVoting", f"Test {tag}: {col} (uncalibrated)"], round(A.loc[key, "auc"], 4)):
            bad.append(f"{tag} {col}")
check("E tables", "S3 ablation test AUROCs = test_model_auc files", not bad, bad)
S5 = rd(TAB + "S5_complete_case.csv").set_index("arm")
bad = [a for a in ARMS if not np.isclose(S5.loc[a, "auc_complete_case_model"],
                                         round(rd(P2 + f"test_complete_case_{a}.csv").auc_complete_case_model.iloc[0], 3),
                                         equal_nan=True)]
check("E tables", "S5 complete-case AUROC = NB3 file", not bad, bad)
S6 = rd(TAB + "S6_hyperparameters.csv")
bad = []
for arm in ARMS:
    tj = json.load(open(M2 + f"tuning_{arm}.json"))["best_params"]
    for mname, prm in tj.items():
        s = S6[(S6.arm == arm) & (S6.model == mname)].hyperparameters.iloc[0]
        for k, v in prm.items():
            if f"{k}={round(v, 4) if isinstance(v, float) else v}" not in s:
                bad.append(f"{arm} {mname} {k}")
check("E tables", "S6 hyperparameters = tuning files", not bad, bad[:5])
S7 = rd(TAB + "S7_final_features.csv")
bad = [a for a in ARMS if set(S7[S7.arm == a].feature) != set(joblib.load(M2 + f"final_{a}.joblib")["features"])]
check("E tables", "S7 feature list = features inside the saved final models", not bad, bad)
S9 = rd(TAB + "S9_multiview.csv")
bad = []
for arm in ARMS:
    A = rd(P2 + f"test_model_auc_{arm}.csv").set_index("model")
    r = S9[(S9.arm == arm) & (S9.view == "Multi-View meta-learner")].iloc[0]
    k = A.loc["MultiView_meta"]
    if r["AUROC (95% CI)"] != f"{k.auc:.3f} ({k.lo:.3f}-{k.hi:.3f})":
        bad.append(arm)
check("E tables", "S9 Multi-View AUROC = test_model_auc files", not bad, bad)
T3 = rd(OUT4 + "table3_risk_tiers.csv"); TR = rd(P2 + "test_risk_tiers.csv")
mrg = T3.merge(TR, on=["arm", "tier"], suffixes=("", "_nb3"))
check("E tables", "Table 3 tier counts = NB3 test_risk_tiers.csv",
      (mrg.n == mrg.n_nb3).all() and (mrg.deaths == mrg.deaths_nb3).all())
T6 = rd(OUT4 + "table6_delirium24h_deployment.csv"); Hd = rd(P2 + "test_headline_metrics.csv").set_index("arm")
check("E tables", "Table 6 'All test patients' AUROC and CI = headline metrics",
      all(np.isclose(T6[(T6.arm == a) & (T6.group == "All test patients")].auc_final.iloc[0], Hd.loc[a, "auc"]) and
          np.isclose(T6[(T6.arm == a) & (T6.group == "All test patients")].auc_lo.iloc[0], Hd.loc[a, "auc_lo"])
          for a in ARMS))
SG = rd(P2 + "test_subgroups.csv"); HET = rd(OUT4 + "subgroup_heterogeneity.csv")
bad = []
for _, h in HET.iterrows():
    g = SG[(SG.arm == h.arm) & (SG.subgroup == h.subgroup)].dropna(subset=["auc"])
    se = (g.auc_hi - g.auc_lo) / 3.92; g, se = g[se > 0], se[se > 0]
    w = 1 / se ** 2; pooled = (w * g.auc).sum() / w.sum(); Q = (w * (g.auc - pooled) ** 2).sum()
    if not np.isclose(stats.chi2.sf(Q, len(g) - 1), h.p_heterogeneity):
        bad.append(f"{h.arm} {h.subgroup}")
check("E tables", "Cochran's Q heterogeneity p-values recomputed", not bad, bad)
cvs = rd(P2 + "cv/cv_summary.csv").query("pool == 'B_primary'").set_index("model")
RI = rd(OUT4 + "sample_size_riley.csv")
check("E tables", "Riley: assumed C = nested-CV Soft-Voting AUROC",
      all(np.isclose(RI.assumed_C, round(cvs.loc["SoftVoting", "auc_mean"], 3))))
check("E tables", "Riley: parameters P = number of final features",
      all(RI.set_index("arm").loc[a, "parameters_P"] == len(joblib.load(M2 + f"final_{a}.joblib")["features"]) for a in ARMS))

# ------------------------------------------- F -------------------------------------------
FIGS = ["fig1_flow_diagram", "fig2_roc_pr", "fig3_calibration", "fig4_decision_curve", "fig5_shap",
        "fig6_dependence_pdp", "fig7_risk_tiers_uncertainty", "fig8_subgroups_forest",
        "figS2_shap_temporal", "figS3_gcs_paradox"]
try:
    from PIL import Image
except ImportError:
    Image = None
for fname in FIGS:
    png, pdf = FIG + fname + ".png", FIG + fname + ".pdf"
    ok = os.path.exists(png) and os.path.exists(pdf) and os.path.getsize(pdf) > 5_000
    det = ""
    if ok and Image is not None:
        im = Image.open(png); dpi = im.info.get("dpi", (0, 0))[0]
        ok = ok and round(dpi) >= 299
        det = f"{im.size[0]}x{im.size[1]} px, {round(dpi)} dpi"
    check("F figures", f"{fname}: PNG (>=300 dpi) + vector PDF present", ok, det)

# ------------------------------------------- G -------------------------------------------
ID = {"stay_id", "subject_id", "hadm_id"}
plevel = []
for f in sorted(glob.glob(OUT4 + "**/*.csv", recursive=True)):
    cols = set(rd(f, nrows=0).columns)
    if cols & ID:
        plevel.append({"file": os.path.relpath(f, ".."), "id_columns": ", ".join(sorted(cols & ID))})
PL = pd.DataFrame(plevel, columns=["file", "id_columns"])
PL.to_csv(VER + "patient_level_files_DO_NOT_SHARE.csv", index=False)
public = [p for p in PL.file if "/tables/" in p or "table" in os.path.basename(p)]
check("G privacy", "no paper table (Tables 1-6, S1-S9) contains patient IDs", not public, public)
check("G privacy", "patient-level NB4 files listed (keep private, never push)", True,
      "; ".join(PL.file) or "none")
xl = TAB + "all_tables.xlsx"
if os.path.exists(xl):
    sheets = pd.read_excel(xl, sheet_name=None, nrows=0)
    check("G privacy", "all_tables.xlsx has no patient-ID columns",
          not any(set(v.columns) & ID for v in sheets.values()))

# ------------------------------------------- report -------------------------------------------
REP = pd.DataFrame(R)
REP.to_csv(VER + "V2_audit.csv", index=False)
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 120)
print(f"VERIFICATION PASS 2 - leakage / timeline / consistency / privacy: {(REP.result == 'PASS').sum()} PASS, "
      f"{(REP.result == 'FAIL').sum()} FAIL\n")
print(REP[["section", "check", "result"]].to_string(index=False))
if (REP.result == "FAIL").any():
    print("\nFAILED CHECKS (details):")
    print(REP[REP.result == "FAIL"][["section", "check", "detail"]].to_string(index=False))
print("\nPatient-level files (NEVER share, upload or push):")
print(PL.to_string(index=False) if len(PL) else "  none")
print("\nSaved: nb4/verification/V2_audit.csv, nb4/verification/patient_level_files_DO_NOT_SHARE.csv")

In [ ]:
# ========= NB4 CELL 14: VERIFICATION PASS 3 - reproducibility (fresh kernel, run everything) =======
# Angle: "does the whole of Notebook 4 give exactly the same results from a clean start?"
# Hidden state (a variable left over from an older cell, a cell run out of order) is a classic
# source of wrong numbers. This cell fingerprints every NB4 result table:
#   1st run  (now, after all cells have been run once): saves a reference snapshot
#   2nd run  (after Kernel -> Restart & Run All): compares every result with the reference
# Tables are compared by VALUE (rounded to 10 decimals), so harmless file differences do not count;
# figures are not compared (image files contain timestamps) - they are rebuilt from these tables.
# Output: nb4/verification/V3_snapshot_reference.json (first run), nb4/verification/V3_reproducibility.csv
import os, glob, json, hashlib, datetime, importlib
import numpy as np, pandas as pd
import v2_nb4
importlib.reload(v2_nb4)
from v2_nb4 import OUT4, freeze_check

VER = OUT4 + "verification/"
os.makedirs(VER, exist_ok=True)
REF = VER + "V3_snapshot_reference.json"
n_frozen = freeze_check()


def table_hash(path):
    d = pd.read_csv(path, float_precision="round_trip")
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].round(10)
    return hashlib.md5(d.to_csv(index=False).encode()).hexdigest()


files = sorted(f for f in glob.glob(OUT4 + "**/*.csv", recursive=True) if "/verification/" not in f)
snap = {os.path.relpath(f, OUT4): table_hash(f) for f in files}
snap["_frozen_manifest"] = hashlib.md5(open(OUT4 + "frozen_manifest.json", "rb").read()).hexdigest()

if not os.path.exists(REF):
    json.dump({"created": datetime.datetime.now().isoformat(timespec="seconds"), "hashes": snap},
              open(REF, "w"), indent=1)
    print(f"Frozen files unchanged: {n_frozen}")
    print(f"REFERENCE SNAPSHOT SAVED: {len(snap) - 1} result tables fingerprinted.")
    print("\nNEXT: Kernel -> Restart & Run All. When this cell runs again at the end it will compare.")
else:
    ref = json.load(open(REF))
    rows = []
    for k in sorted(set(ref["hashes"]) | set(snap)):
        a, b = ref["hashes"].get(k), snap.get(k)
        status = ("IDENTICAL" if a == b else "MISSING NOW" if b is None
                  else "NEW (not in reference)" if a is None else "DIFFERENT")
        rows.append({"table": k, "status": status})
    REP = pd.DataFrame(rows)
    REP.to_csv(VER + "V3_reproducibility.csv", index=False)
    n_id = (REP.status == "IDENTICAL").sum()
    print(f"Frozen files unchanged: {n_frozen}")
    print(f"VERIFICATION PASS 3 - reproducibility vs reference of {ref['created']}: "
          f"{n_id} of {len(REP)} result tables IDENTICAL")
    if (REP.status != "IDENTICAL").any():
        print("\nNOT identical (send me this list):")
        print(REP[REP.status != "IDENTICAL"].to_string(index=False))
    else:
        print("PASS: a clean restart reproduces every NB4 result exactly.")
    print("\nSaved: nb4/verification/V3_reproducibility.csv")

In [ ]:
# ========= NB4 CELL 15: build the PUBLIC GitHub release folder (code + aggregate results only) =====
# TRIPOD+AI item 18f (code sharing) and item 22 (model specification / rebuild).
# Builds ../github_release/ from scratch every time it runs:
#   notebooks/  the 4 notebooks with ALL outputs removed (outputs can contain patient-level prints)
#   src/        the v2_*.py helper modules
#   results/    aggregate result tables only (whitelist), figures/ (paper figures)
#   README.md, requirements.txt (exact versions used), LICENSE (MIT), .gitignore, CITATION.cff
# NOT included (PhysioNet data use agreement): any MIMIC data, any patient-level file
# (predictions, prepared data, SHAP per patient), trained model files (.joblib).
# The old v1 notebooks (notebook3.ipynb, notebook4.ipynb) are superseded and NOT included.
# (copies of this cell inside Notebook 4 are skipped by the scan: they contain the scan rules themselves)
# A final scan STOPS the cell if anything patient-level, any credential, or any notebook output
# is found in the release folder. Nothing in the project itself is changed.
import os, re, json, glob, shutil, datetime, importlib
import pandas as pd
import v2_nb4
importlib.reload(v2_nb4)
from v2_nb4 import P2, M2, FIG, OUT4, freeze_check

n_frozen = freeze_check()
REL = "../github_release/"
REPO_NAME = "mimic-iv-cancer-delirium-mortality"
NOTEBOOKS = {"1st notebook.ipynb": "NB1_cohort.ipynb",                  # your file -> name in the repo
             "Notebook2.ipynb": "NB2_features_TPIFE.ipynb",
             "Notebook3_v2.ipynb": "NB3_modelling_validation.ipynb",
             "Notebook4_v2.ipynb": "NB4_evaluation_explainability.ipynb"}
missing = [k for k in NOTEBOOKS if not os.path.exists(k)]
assert not missing, (f"Notebook file(s) not found in this folder: {missing}. Notebooks here: "
                     f"{sorted(glob.glob('*.ipynb'))} - tell me the right names.")

if os.path.exists(REL):
    shutil.rmtree(REL)
for d in ["notebooks", "src", "results/tables", "results/nb3", "results/verification", "figures"]:
    os.makedirs(REL + d, exist_ok=True)

# ------------------------------- 1. notebooks without outputs -------------------------------
ID_PATTERN = re.compile(r"(?<![\d.])[123]\d{7}(?![\d.])")   # MIMIC subject_id / hadm_id / stay_id
redacted = {}
for src, dst in NOTEBOOKS.items():
    nb = json.load(open(src, encoding="utf-8"))
    for c in nb["cells"]:
        if c.get("cell_type") == "code":
            c["outputs"], c["execution_count"] = [], None
        c["metadata"] = {}
        c.pop("attachments", None)                       # pasted images in markdown cells
    nb.get("metadata", {}).pop("widgets", None)
    # remove individual MIMIC record identifiers (8-digit subject/hadm/stay ids) that old, superseded
    # spot-check cells typed into the code; each distinct id becomes REDACTED_ID_<k>
    found = {}
    for c in nb["cells"]:
        src_txt = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        for hit in ID_PATTERN.findall(src_txt):
            found.setdefault(hit, f"REDACTED_ID_{len(found) + 1}")
        new = ID_PATTERN.sub(lambda m: found[m.group(0)], src_txt)
        c["source"] = new.splitlines(keepends=True)
    if found:
        redacted[dst] = len(found)
    json.dump(nb, open(REL + "notebooks/" + dst, "w", encoding="utf-8"), indent=1, ensure_ascii=False)

# ------------------------------------ 2. helper modules ------------------------------------
mods = sorted(glob.glob("v2_*.py"))
for f in mods:
    shutil.copy(f, REL + "src/" + f)

# ------------------------------ 3. aggregate results (whitelist) ------------------------------
NB4_FILES = ["table1_baseline.csv", "table_s1_split_balance.csv", "calibration_metrics.csv",
             "decision_curve_all.csv", "table3_risk_tiers.csv", "table4_uncertainty_tiers.csv",
             "table5_decision_pathway.csv", "table6_delirium24h_deployment.csv", "tier_trend_and_uncertainty.csv",
             "subgroup_heterogeneity.csv", "shap_importance_random_split.csv", "shap_importance_temporal_split.csv",
             "shap_interactions_xgb_top20.csv", "pdp_top6_random_split.csv", "sample_size_riley.csv",
             "tripod_ai_checklist.csv", "posthoc_model_comparisons.csv", "gcs_paradox_models.csv",
             "gcs_paradox_groups.csv", "fig1_flow_counts.csv", "software_versions.json"]
NB3_FILES = ["test_headline_metrics.csv", "test_model_auc_random_split.csv", "test_model_auc_temporal_split.csv",
             "test_delong_random_split.csv", "test_delong_temporal_split.csv", "test_subgroups.csv",
             "test_risk_tiers.csv", "test_net_benefit_random_split.csv", "test_net_benefit_temporal_split.csv",
             "test_complete_case_random_split.csv", "test_complete_case_temporal_split.csv",
             "selected_features_random_split.csv", "selected_features_temporal_split.csv",
             "cv/cv_summary.csv", "cv/cv_corrected_ttests.csv", "cv/cv_feature_selection_frequency.csv",
             "pilot_labs/pilot_cv_auc.csv", "tpife_thresholds.csv", "attrition_counts_final.csv",
             "icd_code_appendix.csv", "icd_code_exclusions.csv", "nb3_verification.csv"]
M2_FILES = ["tuning_random_split.json", "tuning_temporal_split.json", "locked_rules_random_split.json",
            "locked_rules_temporal_split.json", "feature_pool_decision.json"]
copied, skipped = [], []
for f in NB4_FILES:
    (shutil.copy(OUT4 + f, REL + "results/" + f), copied.append(f)) if os.path.exists(OUT4 + f) else skipped.append(f)
for f in glob.glob(OUT4 + "tables/*"):
    shutil.copy(f, REL + "results/tables/"); copied.append("tables/" + os.path.basename(f))
for f in ["V1_recompute.csv", "V2_audit.csv", "V3_reproducibility.csv", "paper_numbers.csv"]:
    p = OUT4 + "verification/" + f
    (shutil.copy(p, REL + "results/verification/" + f), copied.append(f)) if os.path.exists(p) else skipped.append(f)
for f in NB3_FILES:
    p = P2 + f
    (shutil.copy(p, REL + "results/nb3/" + os.path.basename(f)), copied.append(f)) if os.path.exists(p) else skipped.append(f)
for f in M2_FILES:
    p = M2 + f
    (shutil.copy(p, REL + "results/nb3/" + f), copied.append(f)) if os.path.exists(p) else skipped.append(f)

# ------------------------------------------ 4. figures ------------------------------------------
FIGS = ["fig1_flow_diagram", "fig2_roc_pr", "fig3_calibration", "fig4_decision_curve", "fig5_shap",
        "fig6_dependence_pdp", "fig7_risk_tiers_uncertainty", "fig8_subgroups_forest",
        "figS2_shap_temporal", "figS3_gcs_paradox"]
for f in FIGS:
    for ext in (".png", ".pdf"):
        shutil.copy(FIG + f + ext, REL + "figures/" + f + ext)

# ----------------------------- 5. README, requirements, licence -----------------------------
sv = json.load(open(OUT4 + "software_versions.json"))
PIP = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn", "xgboost": "xgboost",
       "catboost": "catboost", "lightgbm": "lightgbm", "optuna": "optuna", "imblearn": "imbalanced-learn",
       "comorbidipy": "comorbidipy", "matplotlib": "matplotlib", "shap": "shap", "statsmodels": "statsmodels"}
req = [f"{PIP[k]}=={v}" for k, v in sv.items() if k in PIP] + ["joblib", "openpyxl", "jupyter"]
open(REL + "requirements.txt", "w").write("# exact versions used for the paper (Python "
                                          f"{sv.get('python', '3.13')})\n" + "\n".join(req) + "\n")
year = datetime.date.today().year
open(REL + "LICENSE", "w").write(f"""MIT License

Copyright (c) {year} Devansh Kalwani and Arya

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated
documentation files (the "Software"), to deal in the Software without restriction, including without limitation
the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to
permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of
the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE
WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR
COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR
OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
""")
open(REL + ".gitignore", "w").write("# never commit MIMIC data, patient-level outputs or trained models\n"
                                    "data/\nmodels_v2/\n*.csv.gz\n*.joblib\n*.pkl\n.ipynb_checkpoints/\n"
                                    "__pycache__/\n.DS_Store\n")
open(REL + "CITATION.cff", "w").write(f"""cff-version: 1.2.0
message: "If you use this code, please cite the accompanying article (citation will be added on publication)."
title: "Interpretable 24-hour prediction of 28-day mortality in ICU patients with cancer and delirium (MIMIC-IV)"
authors:
  - family-names: Kalwani
    given-names: Devansh
  - given-names: Arya
date-released: {datetime.date.today().isoformat()}
license: MIT
""")
README = f"""# 28-day mortality prediction in ICU patients with cancer and delirium (MIMIC-IV v3.1)

Code and aggregate results for the article *(title and DOI added on publication)*.
The analysis follows TRIPOD+AI (Collins et al., BMJ 2024;385:e078378).

## What is in this repository
| Folder | Content |
|---|---|
| `notebooks/` | The four analysis notebooks, in run order (outputs removed) |
| `src/` | Helper modules imported by the notebooks (`v2_*.py`) |
| `results/` | Aggregate result tables: paper tables (`tables/`), NB3 test/CV summaries (`nb3/`), verification reports (`verification/`) |
| `figures/` | All paper figures (300-dpi PNG + vector PDF) |

Pipeline: **NB1** cohort (cancer + delirium, first ICU stay, alive and in ICU at 24 h; 2,777 patients) ->
**NB2** first-24-h features incl. the TPIFE temporal framework (391 candidates) ->
**NB3** training-only preparation, stability SHAP-RFE feature selection, Optuna tuning, nested 5x5
cross-validation, pre-specified model choice (Soft-Voting XGBoost + random forest + CatBoost, isotonic
calibration, Cross Venn-Abers uncertainty), locked thresholds/risk tiers, one-time test evaluation on a random
80/20 split and a temporal split (<=2017 / >2017) ->
**NB4** figures, tables, SHAP explanations, risk tiers, subgroup/fairness analysis, sample size (Riley),
three independent verification passes.

NB1 and NB2 are the complete working record and also contain earlier, superseded steps; the final pipeline is
the one that writes to the `*_v2` folders (`data/processed_v2/`, `models_v2/`, `figures_v2/`). Individual record
identifiers typed into superseded spot-check cells have been replaced by `REDACTED_ID_<k>` (those cells are not
part of the final pipeline).

## Data access
No MIMIC data are included. MIMIC-IV v3.1 is available to credentialed users from PhysioNet
(https://physionet.org/content/mimiciv/) after completing the required training and signing the data use agreement.
Place the raw `.csv.gz` files in `data/raw/` and run the notebooks in order from the `notebooks/` folder
(copy `src/*.py` next to the notebooks). Outputs are written to `data/processed_v2/`, `models_v2/`, `figures_v2/`.

## Trained models
The trained model files are not posted publicly because they were fitted on credentialed patient data.
Everything needed to rebuild them exactly is here: the code, the selected features
(`results/nb3/selected_features_*.csv`), the tuned hyperparameters (`results/nb3/tuning_*.json`,
`results/tables/S6_hyperparameters.csv`), fixed random seeds, and the locked decision threshold and risk-tier
cut-offs (`results/nb3/locked_rules_*.json`). Credentialed MIMIC-IV users may request the fitted model files
from the authors.

## Environment
Python {sv.get('python', '')}; exact package versions in `requirements.txt`.

## Licence
Code: MIT (see `LICENSE`). Use of MIMIC-IV data is governed by the PhysioNet credentialed data use agreement.
"""
open(REL + "README.md", "w", encoding="utf-8").write(README)

# --------------------------------------- 6. safety scan ---------------------------------------
SELF = "NB4 CELL 15"          # copies of this cell (old or new) contain the scan rules themselves -> skipped


def text_of(f):
    """Text to scan. For notebooks: the code/markdown of every cell except copies of this cell."""
    if f.endswith(".ipynb"):
        nb = json.load(open(f, encoding="utf-8"))
        parts = []
        for c in nb["cells"]:
            s_ = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
            if SELF not in s_[:200]:
                parts.append(s_)
        return "\n".join(parts)
    return open(f, encoding="utf-8", errors="ignore").read()


def where(txt, pat):
    """Short, masked snippet of the first matching line (so you can find it without exposing it)."""
    m = re.search(pat, txt)
    line = txt[txt.rfind("\n", 0, m.start()) + 1: txt.find("\n", m.end()) if txt.find("\n", m.end()) > 0 else None]
    return re.sub(r"(?i)(pass[w]ord|token|secret|key)(\s*[=: ]\s*)\S+", r"\1\2****", line.strip())[:90]


problems = []
ID = {"stay_id", "subject_id", "hadm_id"}
for f in glob.glob(REL + "**/*", recursive=True):
    if os.path.isdir(f):
        continue
    ext = os.path.splitext(f)[1].lower()
    if ext in (".gz", ".joblib", ".pkl", ".parquet", ".feather", ".npy", ".npz"):
        problems.append(f"data/model file type not allowed: {f}")
    if ext == ".csv":
        d = pd.read_csv(f)
        if set(d.columns) & ID:
            problems.append(f"patient-ID column in {f}")
        if {"y", "mortality_28d"} & set(d.columns) and len(d) > 100:
            problems.append(f"looks patient-level (outcome column, {len(d)} rows): {f}")
    if ext == ".ipynb":
        nb = json.load(open(f, encoding="utf-8"))
        if any(c.get("outputs") for c in nb["cells"] if c.get("cell_type") == "code"):
            problems.append(f"notebook still has outputs: {f}")
    if ext in (".ipynb", ".py", ".md", ".txt", ".json", ".cff"):
        txt = text_of(f)
        for pat, what in [(r"(?i)pass[w]ord\s*[=:]\s*['\"][^'\"]+['\"]", "password"),
                          (r"(?i)w[g]et[^\n]*--pass[w]ord", "PhysioNet password in wget"),
                          (r"(?i)(api[_-]?key|secret|token)\s*[=:]\s*['\"][A-Za-z0-9_\-]{12,}['\"]", "key/token"),
                          (r"gh[pousr]_[A-Za-z0-9]{30,}", "GitHub token")]:
            if re.search(pat, txt):
                problems.append(f"possible {what} in {f}  ->  line: {where(txt, pat)}")
paths = set()
for f in glob.glob(REL + "notebooks/*.ipynb") + glob.glob(REL + "src/*.py"):
    txt = text_of(f)
    paths |= set(re.findall(r"(/Use[r]s/[^/\s'\"\[]+|/hom[e]/[^/\s'\"\[]+|[A-Z]:\\\\Use[r]s\\\\[^\\\s'\"\[]+)", txt))
ids_in_code = {}
for f in glob.glob(REL + "notebooks/*.ipynb") + glob.glob(REL + "src/*.py"):
    txt = text_of(f)
    hits = sorted(set(re.findall(r"(?<![\d.])[123]\d{7}(?![\d.])", txt)))   # MIMIC subject/hadm/stay id pattern
    if hits:
        ids_in_code[os.path.basename(f)] = hits[:8]

size_mb = sum(os.path.getsize(f) for f in glob.glob(REL + "**/*", recursive=True) if os.path.isfile(f)) / 1e6
print(f"Frozen files unchanged: {n_frozen}")
print(f"Release folder built: {os.path.abspath(REL)}  ({size_mb:.1f} MB)")
print(f"  notebooks: {len(NOTEBOOKS)} (outputs removed) | modules: {len(mods)} ({', '.join(mods)})")
print(f"  result files: {len(copied)} | figures: {2 * len(FIGS)}")
if redacted:
    print(f"  record identifiers removed from the public notebooks: {redacted}")
if skipped:
    print(f"  not found, skipped (OK if they never existed): {skipped}")
if paths:
    print(f"  NOTE: personal folder paths appear in the code (not secret, but visible): {sorted(paths)}")
if ids_in_code:
    problems.append(f"8-digit numbers that look like MIMIC record IDs still in the code: {ids_in_code}")
if problems:
    print("\nSAFETY SCAN FAILED - DO NOT PUSH. Send me this list:")
    for p in problems:
        print("  -", p)
else:
    print("\nSAFETY SCAN PASSED: no patient-level data, no model files, no notebook outputs, no credentials.")